# DMET-SSTKG — Decentralized Comparative Analysis on the Cholec80/M2CAI Remote Interface

This notebook rebuilds the comparative experiment from the five retained study artifacts:

1. the source S-STKG binary cache;
2. the remote S-STKG binary cache;
3. the case-disjoint source split;
4. the phase-initialized motif-extractor checkpoint; and
5. the serialized Dirichlet-conformal source hub.

A single frozen graph-temporal representation is extracted once and reused by every method. The comparative set contains the source-only reference, FedAvg, SCAFFOLD, FedProto, FedTGP, D-PSGD ring gossip, and DMET-SSTKG direct consultation. Training-based references optimize the same residual adapter and phase head so that the comparison isolates the collaboration protocol rather than a different backbone.

The notebook produces the comparative performance table, seed-level result data, communication accounting, and the seed-robustness/payload figure. The implementations preserve the central algorithmic mechanism of each cited method while adapting it to the common frozen S-STKG feature interface used in this study.

## 1. Environment

Use Python 3.9 or later. Install PyTorch for the local CPU or CUDA environment, followed by the remaining packages:

```bash
pip install numpy pandas scikit-learn matplotlib tqdm jupyter nbformat
pip install torch-geometric
```

The notebook does not download datasets, checkpoints, or hub artifacts.

In [ ]:
import importlib

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "torch": "torch",
    "torch_geometric": "torch-geometric",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        module = importlib.import_module(module_name)
        print(f"[OK] {module_name}: {getattr(module, '__version__', 'imported')}")
    except Exception as exc:
        missing.append(package_name)
        print(f"[MISSING] {module_name}: {exc}")

if missing:
    raise RuntimeError(
        "Install the missing packages before continuing: " + ", ".join(sorted(set(missing)))
    )

## 2. Local Artifact Paths

Enter all six paths explicitly. No input or output location is inferred from the notebook directory, the current working directory, or a repository layout.

- `SOURCE_BINARY_CACHE`: source `.pt` fragment cache generated by `Codes/Hub_Construction/11_source_motif_memory_dirichlet_conformal_hub.ipynb`, or use the pre-generated `Binary_Cache_HUB_Cholec80_M2cai/` directory included in the repository;
- `REMOTE_BINARY_CACHE`: held-out Cholec80/M2CAI remote `.pt` fragment cache generated by `Codes/Remote_Update_Cholecystectomy/12_same_ontology_cholec80_m2cai_remote_update.ipynb`, or use the pre-generated `Binary_Cache_Remote_Cholec80_M2cai/` directory included in the repository;
- `SOURCE_SPLIT_NPZ`: case-disjoint source split containing `train_idx`, `val_idx`, and `test_idx`, generated by `Codes/Phase_Classification/08_source_sstkg_phase_classifier.ipynb`, or use the pre-generated `case_level_split_seed42.npz` file included in the repository;
- `BASE_MOTIF_CHECKPOINT`: phase-initialized motif-extractor checkpoint (`best_motif_model.pt` or `best_phase_initialized_motif_extractor.pt`) generated by `Codes/Motif_Extraction/09_phase_initialized_sstkg_motif_extractor.ipynb`;
- `DMET_HUB`: serialized eight-phase Dirichlet-conformal motif hub (`Shareable_Motif_Hub_DIRICHLET_CONFORMAL_PROTOTYPE.pt`) generated by `Codes/Hub_Construction/11_source_motif_memory_dirichlet_conformal_hub.ipynb`;
- `OUTPUT_DIR`: writable directory in which this notebook stores the frozen features and final comparative-analysis outputs.

In [ ]:
from pathlib import Path

SOURCE_BINARY_CACHE = None
REMOTE_BINARY_CACHE = None
SOURCE_SPLIT_NPZ = None
BASE_MOTIF_CHECKPOINT = None
DMET_HUB = None
OUTPUT_DIR = "..."

# Full study protocol.
SEEDS = (42, 43, 44)
SELECTION_SEED = 42
NUM_CLIENTS = 4
COMMUNICATION_ROUNDS = 50
CPU_THREADS = 8

# "retained" runs one source-validation-selected setting per method.
# "compact_grid" searches the small predefined grid on SELECTION_SEED and
# reuses the selected setting for the remaining seeds.
HYPERPARAMETER_MODE = "retained"

# A complete internal feature cache is reused only when its input-artifact
# fingerprints match the paths above.
REBUILD_INTERNAL_FEATURES = False

# Set True only for a short installation test. It reduces rounds, seeds,
# partition trials, and FedTGP server epochs without changing method logic.
FAST_DEVELOPMENT_MODE = False

## 3. Comparative Protocol and References

All methods receive the same frozen 384-dimensional S-STKG fragment representation and begin from the same source phase head within a seed.

| Method | Protocol retained in this notebook |
|---|---|
| Source-only | Frozen source phase head applied directly to the remote fragments. |
| FedAvg [1] | Client-local SGD followed by sample-count-weighted averaging of the adapter/head state. |
| SCAFFOLD [2] | Client and server control variates correct drift during local SGD; full-participation server updates use uniform averaging. |
| FedProto [3] | Clients exchange class prototypes; the server performs class-count-weighted aggregation; local learning includes prototype MSE regularization. |
| FedTGP [4] | Uploaded client prototypes train persistent server-side global prototypes using an adaptive-margin objective. |
| D-PSGD [5] | Four clients perform symmetric ring mixing and local stochastic-gradient updates at every step without a parameter server. |
| DMET-SSTKG | One maximum-entropy motif per remote fragment queries the Dirichlet-calibrated conformal source hub. |

FedProto and FedTGP use client-wise prototype inference on the unseen remote cohort. D-PSGD uses the probability ensemble of the selected peer models. These deployment rules avoid using remote labels or remote adaptation to construct the compared predictions.

**Comparison basis.** All training-based references use the same
case-disjoint source split, four case-wise non-IID clients, frozen
384-dimensional S-STKG representation, source-initialized residual
adapter and phase head, seeds 42--44, and the same held-out
Cholec80/M2CAI remote cohort. Candidate settings and stopping rounds are
selected only from source-validation performance, and remote labels are
attached only after all method predictions have been produced. These are
protocol-adapted reference implementations that preserve each method's
defining optimization and exchanged object, rather than exact
reproductions of the complete native systems.

**Payload accounting.** For the training-based references,
`total_protocol_bytes` includes training communication and the retained
deployment artifact. For DMET-SSTKG, it denotes the online motif
query--response payload; the one-time calibrated hub object is reported
separately and excluded. The values are therefore protocol-specific logical
communication totals rather than stage-matched end-to-end network traffic
measurements.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import io
import json
import math
import os
import random
import shutil
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, log_loss, precision_recall_fscore_support
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool
from tqdm.auto import tqdm
from IPython.display import display

NUM_PHASES = 8
PHASE_NAMES = [
    "Trocar Placement",
    "Preparation",
    "Calot Triangle Dissection",
    "Clipping and Cutting",
    "Gallbladder Dissection",
    "Gallbladder Packaging",
    "Cleaning and Coagulation",
    "Gallbladder Retraction",
]
EPS = 1e-12

METHOD_ORDER = [
    "source_only",
    "fedavg_adapter",
    "scaffold_adapter",
    "fedproto_adapter",
    "fedtgp_adapter",
    "dpsgd_ring_gossip_adapter",
    "dmet_direct",
]

DISPLAY_NAMES = {
    "source_only": "Source-only reference",
    "fedavg_adapter": "FedAvg adapter",
    "scaffold_adapter": "SCAFFOLD adapter",
    "fedproto_adapter": "FedProto adapter",
    "fedtgp_adapter": "FedTGP adapter",
    "dpsgd_ring_gossip_adapter": "D-PSGD ring gossip",
    "dmet_direct": "DMET-SSTKG direct consultation",
}

PROTOCOL_DESCRIPTIONS = {
    "source_only": "Frozen source model applied directly to the remote test split.",
    "fedavg_adapter": "Federated averaging with a global residual adapter and phase head.",
    "scaffold_adapter": "Control-variate federated optimization with a global adapter and phase head.",
    "fedproto_adapter": "Federated prototype sharing with client-wise prototype inference.",
    "fedtgp_adapter": "Trainable global prototypes with server-side adaptive-margin optimization.",
    "dpsgd_ring_gossip_adapter": "Serverless ring-gossip optimization with peer-model probability ensembling.",
    "dmet_direct": "Uncertainty-selected calibrated motif-evidence consultation.",
}

EXCHANGED_INFORMATION = {
    "source_only": "None",
    "fedavg_adapter": "Adapter/head states",
    "scaffold_adapter": "Adapter/head states and control variates",
    "fedproto_adapter": "Class prototypes and deployed client adapters",
    "fedtgp_adapter": "Trainable global prototypes and deployed client adapters",
    "dpsgd_ring_gossip_adapter": "Peer-to-peer adapter/head states",
    "dmet_direct": "Motif-evidence query/response",
}

## 4. Configuration and Artifact Validation

The full protocol uses 50 communication rounds and seeds 42, 43, and 44. The default retained settings correspond to the source-validation choices used for the study workflow. `compact_grid` performs a small source-validation search on seed 42 and locks the selected setting before seeds 43 and 44 are run.

In [ ]:
@dataclass(frozen=True)
class BenchmarkConfig:
    source_binary_cache: Path
    remote_binary_cache: Path
    source_split_npz: Path
    base_motif_checkpoint: Path
    dmet_hub: Path
    output_dir: Path
    seeds: tuple[int, ...]
    selection_seed: int
    num_clients: int
    rounds: int
    cpu_threads: int
    hyperparameter_mode: str
    rebuild_internal_features: bool
    bottleneck_dim: int = 64
    batch_size: int = 64
    inference_batch_size: int = 512
    weight_decay: float = 1e-4
    partition_search_trials: int = 20000
    partition_min_phases_per_client: int = 6
    partition_min_clients_per_phase: int = 2
    partition_max_fragment_ratio: float = 2.0
    fedtgp_server_epochs: int = 1000
    fedtgp_server_batch_size: int = 10
    fedtgp_margin_threshold: float = 100.0
    log_every: int = 5

    @property
    def feature_dir(self) -> Path:
        return self.output_dir / "_internal_frozen_features"

    @property
    def run_dir(self) -> Path:
        return self.output_dir / "_internal_method_runs"


def _required_path(raw: Any, name: str, *, directory: bool) -> Path:
    if raw is None or str(raw).strip() == "":
        raise ValueError(f"{name} is not set in Section 2.")
    path = Path(raw).expanduser().resolve()
    if directory and not path.is_dir():
        raise FileNotFoundError(f"{name} directory was not found: {path}")
    if not directory and not path.is_file():
        raise FileNotFoundError(f"{name} file was not found: {path}")
    return path


def build_config() -> BenchmarkConfig:
    if OUTPUT_DIR is None or str(OUTPUT_DIR).strip() == "":
        raise ValueError("OUTPUT_DIR is not set in Section 2.")
    output_dir = Path(OUTPUT_DIR).expanduser().resolve()

    if HYPERPARAMETER_MODE not in {"retained", "compact_grid"}:
        raise ValueError("HYPERPARAMETER_MODE must be 'retained' or 'compact_grid'.")

    if FAST_DEVELOPMENT_MODE:
        seeds = (42,)
        rounds = 2
        server_epochs = 8
        partition_trials = 200
        log_every = 1
    else:
        seeds = tuple(int(s) for s in SEEDS)
        rounds = int(COMMUNICATION_ROUNDS)
        server_epochs = 1000
        partition_trials = 20000
        log_every = 5

    if int(SELECTION_SEED) not in seeds:
        raise ValueError("SELECTION_SEED must be included in SEEDS.")

    cfg = BenchmarkConfig(
        source_binary_cache=_required_path(SOURCE_BINARY_CACHE, "SOURCE_BINARY_CACHE", directory=True),
        remote_binary_cache=_required_path(REMOTE_BINARY_CACHE, "REMOTE_BINARY_CACHE", directory=True),
        source_split_npz=_required_path(SOURCE_SPLIT_NPZ, "SOURCE_SPLIT_NPZ", directory=False),
        base_motif_checkpoint=_required_path(BASE_MOTIF_CHECKPOINT, "BASE_MOTIF_CHECKPOINT", directory=False),
        dmet_hub=_required_path(DMET_HUB, "DMET_HUB", directory=False),
        output_dir=output_dir,
        seeds=seeds,
        selection_seed=int(SELECTION_SEED),
        num_clients=int(NUM_CLIENTS),
        rounds=rounds,
        cpu_threads=int(CPU_THREADS),
        hyperparameter_mode=str(HYPERPARAMETER_MODE),
        rebuild_internal_features=bool(REBUILD_INTERNAL_FEATURES),
        partition_search_trials=partition_trials,
        fedtgp_server_epochs=server_epochs,
        log_every=log_every,
    )

    if cfg.output_dir.exists() and not cfg.output_dir.is_dir():
        raise ValueError(f"OUTPUT_DIR points to a file rather than a directory: {cfg.output_dir}")

    for input_cache in (cfg.source_binary_cache, cfg.remote_binary_cache):
        if cfg.output_dir == input_cache or input_cache in cfg.output_dir.parents:
            raise ValueError(
                "OUTPUT_DIR must not be the same as, or located inside, an input binary cache: "
                f"{cfg.output_dir}"
            )

    return cfg

## 5. Numerical Utilities and Evaluation Metrics

Communication is counted from the numeric tensors exchanged by each protocol rather than Python serialization overhead. Recognition is evaluated after all public prediction distributions have been created. Macro F1 uses the complete eight-phase label space, matching the comparative table.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def canonical_hash(obj: Any) -> str:
    raw = json.dumps(obj, sort_keys=True, ensure_ascii=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def cache_inventory_hash(directory: Path) -> tuple[str, list[dict[str, Any]]]:
    files = sorted(p for p in Path(directory).iterdir() if p.is_file() and p.suffix == ".pt")
    if not files:
        raise RuntimeError(f"No .pt fragment files were found in {directory}")
    rows = [{"name": p.name, "size": int(p.stat().st_size)} for p in files]
    return canonical_hash(rows), rows


def probability_normalize(values: np.ndarray) -> np.ndarray:
    p = np.asarray(values, dtype=np.float64)
    squeeze = p.ndim == 1
    if squeeze:
        p = p[None, :]
    p = np.clip(p, 0.0, None)
    totals = p.sum(axis=1, keepdims=True)
    bad = totals[:, 0] <= 0
    p = p / np.clip(totals, EPS, None)
    if np.any(bad):
        p[bad] = 1.0 / p.shape[1]
    return p[0] if squeeze else p


def softmax_np(values: np.ndarray, axis: int = -1) -> np.ndarray:
    x = np.asarray(values, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    exp = np.exp(x)
    return exp / np.clip(exp.sum(axis=axis, keepdims=True), EPS, None)


def l2_normalize_np(values: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=np.float32)
    if x.ndim == 1:
        return x / (np.linalg.norm(x) + EPS)
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + EPS)


def normalized_entropy(probabilities: np.ndarray) -> np.ndarray:
    p = probability_normalize(probabilities)
    return -(
        np.clip(p, EPS, 1.0) * np.log(np.clip(p, EPS, 1.0))
    ).sum(axis=1) / np.log(p.shape[1])


def logical_nbytes(obj: Any) -> int:
    if obj is None:
        return 0
    if isinstance(obj, torch.Tensor):
        return int(obj.numel() * obj.element_size())
    if isinstance(obj, np.ndarray):
        return int(obj.nbytes)
    if isinstance(obj, Mapping):
        return int(sum(logical_nbytes(v) for v in obj.values()))
    if isinstance(obj, (list, tuple, set)):
        return int(sum(logical_nbytes(v) for v in obj))
    return 0


def state_logical_nbytes(state: Mapping[str, torch.Tensor]) -> int:
    return int(sum(int(v.numel() * v.element_size()) for v in state.values()))


def classification_metrics(y_true: np.ndarray, probabilities: np.ndarray) -> dict[str, float]:
    y = np.asarray(y_true, dtype=int)
    p = probability_normalize(probabilities)
    pred = p.argmax(axis=1)
    labels = list(range(NUM_PHASES))
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y, pred, labels=labels, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y, pred, labels=labels, average="weighted", zero_division=0
    )
    recalls = []
    for phase in labels:
        mask = y == phase
        if np.any(mask):
            recalls.append(float(np.mean(pred[mask] == phase)))
    try:
        nll = float(log_loss(y, p, labels=labels))
    except ValueError:
        nll = float("nan")
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(np.mean(recalls)) if recalls else float("nan"),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
        "nll": nll,
    }


def better(metrics: Mapping[str, float], best_macro: float, best_accuracy: float) -> bool:
    macro = float(metrics["macro_f1"])
    accuracy = float(metrics["accuracy"])
    return macro > best_macro + 1e-12 or (
        abs(macro - best_macro) <= 1e-12 and accuracy > best_accuracy + 1e-12
    )

## 6. Read-Only S-STKG Caches and Case-Disjoint Source Split

The binary caches are never rewritten. At runtime, node-feature column 7 is set to zero so that the bookkeeping case identifier cannot act as a predictive shortcut. The source split is required to be complete, non-overlapping, and case-disjoint.

In [ ]:
@dataclass(frozen=True)
class SplitView:
    train_idx: np.ndarray
    val_idx: np.ndarray
    test_idx: np.ndarray


class FragmentCache:
    def __init__(self, binary_dir: Path):
        self.binary_dir = Path(binary_dir).expanduser().resolve()
        if not self.binary_dir.is_dir():
            raise FileNotFoundError(self.binary_dir)
        self.files = sorted(p for p in self.binary_dir.iterdir() if p.is_file() and p.suffix == ".pt")
        if not self.files:
            raise RuntimeError(f"No .pt fragments found in {self.binary_dir}")
        self.inventory_sha256, self.inventory_rows = cache_inventory_hash(self.binary_dir)
        self._metadata: pd.DataFrame | None = None

    def __len__(self) -> int:
        return len(self.files)

    @staticmethod
    def _unpack(path: Path) -> tuple[list[Any], dict[str, Any]]:
        obj = torch.load(path, map_location="cpu", weights_only=False)
        if isinstance(obj, dict) and "graph_seq" in obj:
            sequence = obj["graph_seq"]
            metadata = dict(obj.get("metadata", {}) or {})
        elif isinstance(obj, list):
            sequence = obj
            metadata = {}
        else:
            raise TypeError(f"Unsupported fragment object in {path}")
        if not isinstance(sequence, list) or not sequence:
            raise ValueError(f"Empty or invalid graph sequence in {path}")
        metadata["fragment_file"] = path.name
        return sequence, metadata

    def load(self, index: int) -> tuple[list[Any], dict[str, Any]]:
        return self._unpack(self.files[int(index)])

    @staticmethod
    def label_from(sequence: list[Any], metadata: Mapping[str, Any]) -> int:
        for key in ("true_phase_id", "phase_label_id"):
            if key in metadata:
                value = int(metadata[key])
                if 0 <= value < NUM_PHASES:
                    return value
        if sequence and hasattr(sequence[0], "y") and sequence[0].y is not None:
            value = int(torch.as_tensor(sequence[0].y).reshape(-1)[0].item())
            if 0 <= value < NUM_PHASES:
                return value
        raw = metadata.get("raw_phase_label")
        if raw is not None:
            raw = int(raw)
            if 1 <= raw <= NUM_PHASES:
                return raw - 1
            if 0 <= raw < NUM_PHASES:
                return raw
        raise ValueError(f"Cannot infer a zero-based phase label for {metadata.get('fragment_file')}")

    @staticmethod
    def case_from(metadata: Mapping[str, Any]) -> str:
        case_id = str(metadata.get("case_id", "")).strip()
        if not case_id:
            raise ValueError(f"Missing case_id in {metadata.get('fragment_file')}")
        return case_id

    @staticmethod
    def _sanitize_graph(data: Any, *, strip_label: bool) -> Any:
        item = data.clone() if hasattr(data, "clone") else copy.deepcopy(data)
        if not hasattr(item, "x") or item.x is None or item.x.ndim != 2 or item.x.shape[1] < 11:
            raise ValueError("Every graph must contain an [N, 11] node-feature tensor.")
        item.x = item.x.clone()
        item.x[:, 7] = 0.0
        item.is_v = item.x[:, 10] == 1
        if strip_label and hasattr(item, "y"):
            item.y = torch.tensor([-1], dtype=torch.long)
        return item

    def load_for_inference(self, index: int, device: torch.device) -> tuple[list[Any], dict[str, Any]]:
        sequence, metadata = self.load(index)
        clean = [self._sanitize_graph(graph, strip_label=True).to(device) for graph in sequence]
        return clean, {
            "fragment_file": str(metadata.get("fragment_file", self.files[int(index)].name)),
            "case_id": self.case_from(metadata),
            "prefix": str(metadata.get("prefix", "")),
        }

    def metadata(self) -> pd.DataFrame:
        if self._metadata is not None:
            return self._metadata.copy()
        rows = []
        for index, path in enumerate(self.files):
            sequence, metadata = self._unpack(path)
            rows.append({
                "dataset_index": int(index),
                "fragment_file": path.name,
                "case_id": self.case_from(metadata),
                "prefix": str(metadata.get("prefix", "")),
                "true_phase_id": int(self.label_from(sequence, metadata)),
                "num_frames": int(len(sequence)),
            })
        self._metadata = pd.DataFrame(rows).sort_values("dataset_index").reset_index(drop=True)
        return self._metadata.copy()

    def label_table(self) -> pd.DataFrame:
        return self.metadata()[
            ["dataset_index", "fragment_file", "case_id", "prefix", "true_phase_id"]
        ].copy()


def load_source_split(path: Path, dataset_length: int) -> SplitView:
    obj = np.load(Path(path), allow_pickle=False)
    required = {"train_idx", "val_idx", "test_idx"}
    missing = required - set(obj.files)
    if missing:
        raise KeyError(f"Source split is missing keys: {sorted(missing)}")
    arrays = {
        key: np.asarray(obj[key], dtype=np.int64).reshape(-1)
        for key in ("train_idx", "val_idx", "test_idx")
    }
    for name, values in arrays.items():
        if len(values) == 0:
            raise ValueError(f"{name} is empty")
        if len(np.unique(values)) != len(values):
            raise ValueError(f"{name} contains duplicate indices")
        if np.any(values < 0) or np.any(values >= int(dataset_length)):
            raise IndexError(f"{name} contains out-of-range indices")
    sets = {name: set(values.tolist()) for name, values in arrays.items()}
    for first, second in (("train_idx", "val_idx"), ("train_idx", "test_idx"), ("val_idx", "test_idx")):
        overlap = sets[first] & sets[second]
        if overlap:
            raise ValueError(f"Source split overlap between {first} and {second}: {sorted(overlap)[:10]}")
    if set().union(*sets.values()) != set(range(int(dataset_length))):
        raise ValueError("The source split is not a complete partition of the source cache.")
    return SplitView(arrays["train_idx"], arrays["val_idx"], arrays["test_idx"])


def validate_case_disjoint(metadata: pd.DataFrame, split: SplitView) -> dict[str, Any]:
    by_index = metadata.set_index("dataset_index")
    cases = {
        "train": set(by_index.loc[split.train_idx, "case_id"].astype(str)),
        "validation": set(by_index.loc[split.val_idx, "case_id"].astype(str)),
        "test": set(by_index.loc[split.test_idx, "case_id"].astype(str)),
    }
    overlaps = {
        "train_validation": sorted(cases["train"] & cases["validation"]),
        "train_test": sorted(cases["train"] & cases["test"]),
        "validation_test": sorted(cases["validation"] & cases["test"]),
    }
    if any(overlaps.values()):
        raise RuntimeError(f"Case overlap detected in source split: {overlaps}")
    return {
        "train_cases": sorted(cases["train"]),
        "validation_cases": sorted(cases["validation"]),
        "test_cases": sorted(cases["test"]),
        "train_fragments": int(len(split.train_idx)),
        "validation_fragments": int(len(split.val_idx)),
        "test_fragments": int(len(split.test_idx)),
    }

## 7. Frozen Phase-Initialized Motif Extractor and Feature Export

Every method uses features generated by the same frozen source-trained checkpoint. Source labels are retained for source training and validation. Remote graph labels are stripped before inference; evaluator labels are written separately after the public remote features have been finalized.

In [ ]:
class MotifMosaicGNN(nn.Module):
    def __init__(
        self,
        in_channels: int = 11,
        edge_dim: int = 3,
        hidden_dim: int = 128,
        num_phases: int = 8,
        motif_dim: int = 128,
        motif_window: int = 16,
        motif_stride: int = 8,
    ) -> None:
        super().__init__()
        self.hidden_dim = int(hidden_dim)
        self.conv1 = GATv2Conv(in_channels, hidden_dim, edge_dim=edge_dim)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim)
        self.context_attn = nn.Linear(hidden_dim * 2, 1)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 3, num_phases)
        self.motif_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, motif_dim)
        )
        self.motif_cls = nn.Linear(motif_dim, num_phases)
        self.window_fuse = nn.Linear(hidden_dim * 2, hidden_dim)
        self.motif_window = int(motif_window)
        self.motif_stride = int(motif_stride)

    def encode_frames(self, graph_sequence: list[Any]) -> torch.Tensor:
        prepared = []
        for graph in graph_sequence:
            item = graph.clone() if hasattr(graph, "clone") else copy.deepcopy(graph)
            if not hasattr(item, "is_v") or item.is_v is None:
                item.is_v = item.x[:, 10] == 1
            prepared.append(item)
        batch = Batch.from_data_list(prepared)
        x = self.conv1(batch.x, batch.edge_index, batch.edge_attr).relu()
        x = self.conv2(x, batch.edge_index, batch.edge_attr).relu()
        num_frames = len(prepared)
        mean_embedding = global_mean_pool(x, batch.batch, size=num_frames)
        virtual_mask = batch.is_v.bool()
        if bool(virtual_mask.any()):
            virtual_sum = torch.zeros((num_frames, x.shape[1]), dtype=x.dtype, device=x.device)
            virtual_count = torch.zeros((num_frames, 1), dtype=x.dtype, device=x.device)
            virtual_sum.index_add_(0, batch.batch[virtual_mask], x[virtual_mask])
            virtual_count.index_add_(
                0,
                batch.batch[virtual_mask],
                torch.ones((int(virtual_mask.sum()), 1), dtype=x.dtype, device=x.device),
            )
            virtual_mean = virtual_sum / virtual_count.clamp_min(1.0)
            virtual_mean = torch.where(virtual_count > 0, virtual_mean, mean_embedding)
        else:
            virtual_mean = mean_embedding
        gate = torch.sigmoid(self.context_attn(torch.cat([mean_embedding, virtual_mean], dim=-1)))
        return gate * virtual_mean + (1.0 - gate) * mean_embedding

    def apply_temporal_modeling(self, frame_tensor: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        recurrent_output, hidden_state = self.rnn(frame_tensor.unsqueeze(0))
        recurrent_output = recurrent_output.squeeze(0)
        temporal = 0.5 * (
            recurrent_output[:, : self.hidden_dim] + recurrent_output[:, self.hidden_dim :]
        )
        contextualized = 0.5 * frame_tensor + 0.5 * temporal
        final_state = torch.cat([hidden_state[0], hidden_state[1]], dim=-1)
        mean_skip = frame_tensor.mean(dim=0, keepdim=True)
        return contextualized, torch.cat([final_state, mean_skip], dim=-1)

    def pool_window(self, window: torch.Tensor) -> torch.Tensor:
        return self.window_fuse(
            torch.cat([window.mean(dim=0, keepdim=True), window[-1].unsqueeze(0)], dim=-1)
        )

    def extract_motifs(self, contextualized: torch.Tensor) -> torch.Tensor:
        length = int(contextualized.shape[0])
        if length < self.motif_window:
            pooled = self.pool_window(contextualized)
        else:
            pooled = torch.cat(
                [
                    self.pool_window(contextualized[start : start + self.motif_window])
                    for start in range(0, length - self.motif_window + 1, self.motif_stride)
                ],
                dim=0,
            )
        return F.normalize(self.motif_proj(pooled), dim=-1)

    def forward(self, graph_sequence: list[Any]) -> dict[str, torch.Tensor]:
        frames = self.encode_frames(graph_sequence)
        contextualized, sequence_feature = self.apply_temporal_modeling(frames)
        motif_signatures = self.extract_motifs(contextualized)
        return {
            "seq_logits": self.fc(sequence_feature),
            "seq_feature": sequence_feature,
            "motif_logits": self.motif_cls(motif_signatures),
            "motif_sigs": motif_signatures,
        }


def _normalize_checkpoint_keys(state: Mapping[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    prefix_map = {
        "graph_conv_1.": "conv1.",
        "graph_conv_2.": "conv2.",
        "virtual_context_gate.": "context_attn.",
        "temporal_encoder.": "rnn.",
        "phase_head.": "fc.",
        "motif_projection.": "motif_proj.",
        "motif_phase_head.": "motif_cls.",
        "window_fusion.": "window_fuse.",
    }
    normalized = {}
    for raw_key, value in state.items():
        key = str(raw_key)
        if key.startswith("module."):
            key = key[7:]
        for old_prefix, new_prefix in prefix_map.items():
            if key.startswith(old_prefix):
                key = new_prefix + key[len(old_prefix) :]
                break
        normalized[key] = value
    return normalized


def load_motif_model(checkpoint_path: Path, device: torch.device) -> MotifMosaicGNN:
    model = MotifMosaicGNN().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict):
        state = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    else:
        state = checkpoint
    if not isinstance(state, Mapping):
        raise TypeError("Motif checkpoint does not contain a model state dictionary.")
    state = _normalize_checkpoint_keys(state)
    result = model.load_state_dict(state, strict=False)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(
            "Checkpoint/model mismatch: "
            f"missing={result.missing_keys}, unexpected={result.unexpected_keys}"
        )
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def _extract_cache_features(
    model: MotifMosaicGNN,
    cache: FragmentCache,
    device: torch.device,
    *,
    include_labels: bool,
    description: str,
) -> dict[str, Any]:
    fragment_rows = []
    sequence_features = []
    sequence_probabilities = []
    selected_motif_embeddings = []
    selected_motif_probabilities = []

    model.eval()
    with torch.no_grad():
        for index in tqdm(range(len(cache)), desc=description):
            sequence, public = cache.load_for_inference(index, device)
            output = model(sequence)
            seq_feature = output["seq_feature"].detach().cpu().numpy()[0].astype(np.float32)
            seq_probs = torch.softmax(output["seq_logits"], dim=1).detach().cpu().numpy()[0].astype(np.float32)
            motif_embeddings = output["motif_sigs"].detach().cpu().numpy().astype(np.float32)
            motif_probs = torch.softmax(output["motif_logits"], dim=1).detach().cpu().numpy().astype(np.float32)
            entropies = normalized_entropy(motif_probs)
            query_index = int(np.argmax(entropies))

            row = {
                "dataset_index": int(index),
                "case_id": str(public["case_id"]),
                "fragment_file": str(public["fragment_file"]),
                "prefix": str(public.get("prefix", "")),
                "num_motifs": int(len(motif_embeddings)),
                "selected_query_motif_index": int(query_index),
                "selected_query_entropy": float(entropies[query_index]),
                "seq_pred_id": int(np.argmax(seq_probs)),
                "selected_motif_pred_id": int(np.argmax(motif_probs[query_index])),
            }
            if include_labels:
                raw_sequence, raw_metadata = cache.load(index)
                label = int(cache.label_from(raw_sequence, raw_metadata))
                row["true_phase_id"] = label

            fragment_rows.append(row)
            sequence_features.append(seq_feature)
            sequence_probabilities.append(seq_probs)
            selected_motif_embeddings.append(motif_embeddings[query_index])
            selected_motif_probabilities.append(motif_probs[query_index])

    return {
        "records": pd.DataFrame(fragment_rows).sort_values("dataset_index").reset_index(drop=True),
        "sequence_features": np.asarray(sequence_features, dtype=np.float32),
        "sequence_probabilities": probability_normalize(
            np.asarray(sequence_probabilities, dtype=np.float32)
        ).astype(np.float32),
        "selected_motif_embeddings": l2_normalize_np(
            np.asarray(selected_motif_embeddings, dtype=np.float32)
        ),
        "selected_motif_probabilities": probability_normalize(
            np.asarray(selected_motif_probabilities, dtype=np.float32)
        ).astype(np.float32),
    }


def _feature_manifest_inputs(cfg: BenchmarkConfig) -> dict[str, Any]:
    source_inventory, _ = cache_inventory_hash(cfg.source_binary_cache)
    remote_inventory, _ = cache_inventory_hash(cfg.remote_binary_cache)
    return {
        "source_cache_inventory": source_inventory,
        "remote_cache_inventory": remote_inventory,
        "source_split_sha256": sha256_file(cfg.source_split_npz),
        "motif_checkpoint_sha256": sha256_file(cfg.base_motif_checkpoint),
    }


def feature_cache_complete(feature_dir: Path) -> bool:
    expected = [
        "feature_manifest.json",
        "source_records.csv",
        "source_arrays.npz",
        "remote_public_records_NO_LABELS.csv",
        "remote_public_arrays_NO_LABELS.npz",
        "remote_evaluator_labels.csv",
        "base_head_state.npz",
    ]
    return feature_dir.is_dir() and all((feature_dir / name).is_file() for name in expected)


def export_or_load_feature_bundle(cfg: BenchmarkConfig) -> dict[str, Any]:
    feature_dir = cfg.feature_dir
    manifest_path = feature_dir / "feature_manifest.json"
    current_inputs = _feature_manifest_inputs(cfg)

    reuse = False
    if not cfg.rebuild_internal_features and feature_cache_complete(feature_dir):
        stored = json.loads(manifest_path.read_text(encoding="utf-8"))
        reuse = stored.get("input_fingerprints") == current_inputs
        if not reuse:
            print("Internal feature fingerprints changed; rebuilding frozen features.")

    if not reuse:
        if feature_dir.exists():
            shutil.rmtree(feature_dir)
        feature_dir.mkdir(parents=True, exist_ok=True)

        device = torch.device("cpu")
        source_cache = FragmentCache(cfg.source_binary_cache)
        remote_cache = FragmentCache(cfg.remote_binary_cache)
        source_metadata = source_cache.metadata()
        source_split = load_source_split(cfg.source_split_npz, len(source_cache))
        split_audit = validate_case_disjoint(source_metadata, source_split)
        model = load_motif_model(cfg.base_motif_checkpoint, device)

        source = _extract_cache_features(
            model, source_cache, device, include_labels=True, description="source frozen features"
        )
        remote = _extract_cache_features(
            model, remote_cache, device, include_labels=False, description="remote public frozen features"
        )

        split_membership = np.full(len(source["records"]), "", dtype=object)
        split_membership[source_split.train_idx] = "train"
        split_membership[source_split.val_idx] = "validation"
        split_membership[source_split.test_idx] = "test"
        source["records"] = source["records"].copy()
        source["records"]["split"] = split_membership

        source["records"].to_csv(feature_dir / "source_records.csv", index=False)
        np.savez_compressed(
            feature_dir / "source_arrays.npz",
            sequence_features=source["sequence_features"],
            sequence_probabilities=source["sequence_probabilities"],
            selected_motif_embeddings=source["selected_motif_embeddings"],
            selected_motif_probabilities=source["selected_motif_probabilities"],
        )
        remote["records"].to_csv(feature_dir / "remote_public_records_NO_LABELS.csv", index=False)
        np.savez_compressed(
            feature_dir / "remote_public_arrays_NO_LABELS.npz",
            sequence_features=remote["sequence_features"],
            sequence_probabilities=remote["sequence_probabilities"],
            selected_motif_embeddings=remote["selected_motif_embeddings"],
            selected_motif_probabilities=remote["selected_motif_probabilities"],
        )

        # Public features are complete before evaluator labels are read.
        (feature_dir / "remote_public_generation_BEFORE_LABELS.json").write_text(
            json.dumps(
                {
                    "status": "PUBLIC_REMOTE_FEATURES_FINALIZED_BEFORE_LABEL_ACCESS",
                    "num_remote_fragments": int(len(remote["records"])),
                    "public_feature_hash": canonical_hash(
                        {
                            "records": remote["records"].to_dict(orient="records"),
                            "sequence_feature_shape": list(remote["sequence_features"].shape),
                            "motif_feature_shape": list(remote["selected_motif_embeddings"].shape),
                        }
                    ),
                },
                indent=2,
            ),
            encoding="utf-8",
        )
        remote_cache.label_table().to_csv(feature_dir / "remote_evaluator_labels.csv", index=False)
        np.savez_compressed(
            feature_dir / "base_head_state.npz",
            weight=model.fc.weight.detach().cpu().numpy().astype(np.float32),
            bias=model.fc.bias.detach().cpu().numpy().astype(np.float32),
        )

        manifest = {
            "status": "COMPLETE",
            "input_fingerprints": current_inputs,
            "source_fragments": int(len(source["records"])),
            "remote_fragments": int(len(remote["records"])),
            "source_case_split": split_audit,
            "sequence_feature_dim": int(source["sequence_features"].shape[1]),
            "motif_feature_dim": int(source["selected_motif_embeddings"].shape[1]),
            "encoder_frozen": True,
        }
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    else:
        print(f"Reusing validated frozen features: {feature_dir}")

    source_records = pd.read_csv(feature_dir / "source_records.csv")
    source_arrays = np.load(feature_dir / "source_arrays.npz")
    remote_records = pd.read_csv(feature_dir / "remote_public_records_NO_LABELS.csv")
    remote_arrays = np.load(feature_dir / "remote_public_arrays_NO_LABELS.npz")
    base_head = np.load(feature_dir / "base_head_state.npz")

    return {
        "source_records": source_records,
        "source_seq_features": np.asarray(source_arrays["sequence_features"], dtype=np.float32),
        "source_seq_probs": np.asarray(source_arrays["sequence_probabilities"], dtype=np.float32),
        "source_selected_motif_embeddings": np.asarray(
            source_arrays["selected_motif_embeddings"], dtype=np.float32
        ),
        "source_selected_motif_probs": np.asarray(
            source_arrays["selected_motif_probabilities"], dtype=np.float32
        ),
        "remote_records": remote_records,
        "remote_seq_features": np.asarray(remote_arrays["sequence_features"], dtype=np.float32),
        "remote_seq_probs": np.asarray(remote_arrays["sequence_probabilities"], dtype=np.float32),
        "remote_selected_motif_embeddings": np.asarray(
            remote_arrays["selected_motif_embeddings"], dtype=np.float32
        ),
        "remote_selected_motif_probs": np.asarray(
            remote_arrays["selected_motif_probabilities"], dtype=np.float32
        ),
        "remote_labels_path": feature_dir / "remote_evaluator_labels.csv",
        "base_fc_weight": np.asarray(base_head["weight"], dtype=np.float32),
        "base_fc_bias": np.asarray(base_head["bias"], dtype=np.float32),
    }

## 8. Deterministic Case-Wise Non-IID Source Clients

Source-training cases are assigned to four clients without splitting a case across clients. The search preserves usable phase coverage and fragment balance while selecting an assignment with measurable phase-distribution skew. Validation and source-test cases are never assigned to a client.

In [ ]:
@dataclass(frozen=True)
class PartitionResult:
    clients: dict[str, list[str]]
    client_indices: list[np.ndarray]
    diagnostics: pd.DataFrame
    score: float


def _normalize_histogram(values: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=np.float64)
    return x / x.sum() if x.sum() > 0 else np.ones_like(x) / len(x)


def js_divergence(first: np.ndarray, second: np.ndarray) -> float:
    p = _normalize_histogram(first)
    q = _normalize_histogram(second)
    midpoint = 0.5 * (p + q)

    def kl(a: np.ndarray, b: np.ndarray) -> float:
        mask = a > 0
        return float(np.sum(a[mask] * np.log(a[mask] / np.clip(b[mask], EPS, None))))

    return 0.5 * kl(p, midpoint) + 0.5 * kl(q, midpoint)


def build_casewise_non_iid_partition(
    source_records: pd.DataFrame,
    seed: int,
    *,
    num_clients: int,
    search_trials: int,
    min_phases_per_client: int,
    min_clients_per_phase: int,
    max_fragment_ratio: float,
) -> PartitionResult:
    train_meta = source_records[source_records["split"].astype(str) == "train"].copy()
    cases = sorted(train_meta["case_id"].astype(str).unique())
    base, remainder = divmod(len(cases), int(num_clients))
    client_case_sizes = [base + (1 if index < remainder else 0) for index in range(int(num_clients))]

    case_histograms = {}
    case_counts = {}
    for case_id in cases:
        subset = train_meta[train_meta["case_id"].astype(str) == case_id]
        case_histograms[case_id] = np.bincount(
            subset["true_phase_id"].to_numpy(dtype=int), minlength=NUM_PHASES
        ).astype(np.float64)
        case_counts[case_id] = int(len(subset))
    global_histogram = np.sum([case_histograms[case] for case in cases], axis=0)

    rng = np.random.default_rng(int(seed))
    best = None
    for _ in range(int(search_trials)):
        shuffled = cases.copy()
        rng.shuffle(shuffled)
        groups = []
        cursor = 0
        for size in client_case_sizes:
            groups.append(sorted(shuffled[cursor : cursor + size]))
            cursor += size

        client_histograms = [np.sum([case_histograms[case] for case in group], axis=0) for group in groups]
        fragment_counts = np.asarray(
            [sum(case_counts[case] for case in group) for group in groups], dtype=np.float64
        )
        phase_coverage = np.asarray([(hist > 0).sum() for hist in client_histograms])
        clients_per_phase = np.sum(np.vstack(client_histograms) > 0, axis=0)
        ratio = float(fragment_counts.max() / max(fragment_counts.min(), 1.0))
        valid = bool(
            np.all(phase_coverage >= int(min_phases_per_client))
            and np.all(clients_per_phase >= int(min_clients_per_phase))
            and ratio <= float(max_fragment_ratio)
        )
        mean_js = float(np.mean([js_divergence(hist, global_histogram) for hist in client_histograms]))
        quantity_cv = float(fragment_counts.std() / max(fragment_counts.mean(), 1.0))
        score = mean_js - 0.20 * quantity_cv
        if valid and (best is None or score > best[0]):
            best = (score, groups, client_histograms, fragment_counts)

    if best is None:
        raise RuntimeError(
            "No valid case-wise non-IID partition was found under the configured constraints."
        )

    score, groups, client_histograms, fragment_counts = best
    clients = {f"client_{index}": group for index, group in enumerate(groups)}
    client_indices = []
    rows = []
    for index, (name, group) in enumerate(clients.items()):
        subset = train_meta[train_meta["case_id"].astype(str).isin(group)]
        indices = subset["dataset_index"].to_numpy(dtype=int)
        client_indices.append(indices)
        histogram = client_histograms[index]
        row = {
            "client": name,
            "num_cases": int(len(group)),
            "cases": "|".join(group),
            "num_fragments": int(fragment_counts[index]),
            "phase_coverage": int(np.sum(histogram > 0)),
            "js_divergence_from_global": float(js_divergence(histogram, global_histogram)),
        }
        row.update({f"phase_{phase + 1}_fragments": int(histogram[phase]) for phase in range(NUM_PHASES)})
        rows.append(row)

    return PartitionResult(
        clients=clients,
        client_indices=client_indices,
        diagnostics=pd.DataFrame(rows),
        score=float(score),
    )

## 9. Shared Residual Adapter and Training Utilities

The common adapter is initialized as an identity transformation, and its classifier is initialized from the frozen source phase head. This gives every training-based reference the same starting prediction function. The backbone and motif extractor remain frozen.

In [ ]:
class ResidualAdapterHead(nn.Module):
    def __init__(self, input_dim: int, bottleneck_dim: int, num_classes: int) -> None:
        super().__init__()
        self.down = nn.Linear(input_dim, bottleneck_dim)
        self.up = nn.Linear(bottleneck_dim, input_dim)
        self.classifier = nn.Linear(input_dim, num_classes)

    def adapted_feature(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.up(F.relu(self.down(x)))

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        feature = self.adapted_feature(x)
        return {"logits": self.classifier(feature), "feature": feature}


def create_initial_adapter_state(
    *,
    input_dim: int,
    bottleneck_dim: int,
    base_weight: np.ndarray,
    base_bias: np.ndarray,
    seed: int,
) -> dict[str, torch.Tensor]:
    torch.manual_seed(int(seed))
    model = ResidualAdapterHead(input_dim, bottleneck_dim, NUM_PHASES)
    nn.init.kaiming_uniform_(model.down.weight, a=np.sqrt(5.0))
    if model.down.bias is not None:
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(model.down.weight)
        bound = 1.0 / np.sqrt(fan_in) if fan_in > 0 else 0.0
        nn.init.uniform_(model.down.bias, -bound, bound)
    nn.init.zeros_(model.up.weight)
    nn.init.zeros_(model.up.bias)
    with torch.no_grad():
        model.classifier.weight.copy_(torch.tensor(base_weight, dtype=torch.float32))
        model.classifier.bias.copy_(torch.tensor(base_bias, dtype=torch.float32))
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def build_adapter(
    state: Mapping[str, torch.Tensor], input_dim: int, bottleneck_dim: int
) -> ResidualAdapterHead:
    model = ResidualAdapterHead(input_dim, bottleneck_dim, NUM_PHASES)
    model.load_state_dict(dict(state), strict=True)
    return model


def average_states(
    states: Sequence[Mapping[str, torch.Tensor]], weights: Sequence[float] | None = None
) -> dict[str, torch.Tensor]:
    if not states:
        raise ValueError("No model states were supplied.")
    if weights is None:
        weights = [1.0] * len(states)
    total = float(sum(weights))
    normalized = [float(weight) / total for weight in weights]
    return {
        key: sum(
            normalized[index] * states[index][key].detach().cpu()
            for index in range(len(states))
        )
        for key in states[0]
    }


def zero_state_like(state: Mapping[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {key: torch.zeros_like(value.detach().cpu()) for key, value in state.items()}


def subtract_states(
    first: Mapping[str, torch.Tensor], second: Mapping[str, torch.Tensor]
) -> dict[str, torch.Tensor]:
    return {
        key: first[key].detach().cpu().clone() - second[key].detach().cpu()
        for key in first
    }


def add_states(
    first: Mapping[str, torch.Tensor], second: Mapping[str, torch.Tensor]
) -> dict[str, torch.Tensor]:
    return {
        key: first[key].detach().cpu().clone() + second[key].detach().cpu()
        for key in first
    }


def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    batch_size: int,
    seed: int,
    *,
    shuffle: bool,
) -> DataLoader:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return DataLoader(
        TensorDataset(
            torch.tensor(x[indices], dtype=torch.float32),
            torch.tensor(y[indices], dtype=torch.long),
        ),
        batch_size=min(int(batch_size), max(int(len(indices)), 1)),
        shuffle=bool(shuffle),
        drop_last=False,
        generator=generator,
        num_workers=0,
    )


def prototype_mse_loss(
    features: torch.Tensor,
    labels: torch.Tensor,
    prototypes: torch.Tensor | None,
    valid: torch.Tensor | None,
) -> torch.Tensor:
    if prototypes is None or valid is None:
        return torch.zeros((), dtype=features.dtype, device=features.device)
    mask = valid[labels]
    if not bool(mask.any()):
        return torch.zeros((), dtype=features.dtype, device=features.device)
    target = prototypes[labels[mask]].to(dtype=features.dtype, device=features.device)
    return F.mse_loss(features[mask], target, reduction="mean")


def train_local_epochs(
    model: ResidualAdapterHead,
    x: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    *,
    learning_rate: float,
    momentum: float,
    weight_decay: float,
    batch_size: int,
    local_epochs: int,
    seed: int,
    prototypes: np.ndarray | None = None,
    prototype_valid: np.ndarray | None = None,
    prototype_lambda: float = 0.0,
) -> dict[str, float]:
    model.train()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=float(learning_rate),
        momentum=float(momentum),
        weight_decay=float(weight_decay),
    )
    criterion = nn.CrossEntropyLoss()
    proto_tensor = None if prototypes is None else torch.tensor(prototypes, dtype=torch.float32)
    valid_tensor = None if prototype_valid is None else torch.tensor(prototype_valid, dtype=torch.bool)
    total = 0
    loss_sum = 0.0
    for epoch in range(int(local_epochs)):
        for xb, yb in make_loader(
            x, y, indices, batch_size, int(seed) + epoch, shuffle=True
        ):
            output = model(xb)
            cross_entropy = criterion(output["logits"], yb)
            proto_loss = prototype_mse_loss(output["feature"], yb, proto_tensor, valid_tensor)
            loss = cross_entropy + float(prototype_lambda) * proto_loss
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += int(len(yb))
            loss_sum += float(loss.detach().item()) * int(len(yb))
    return {"loss": loss_sum / max(total, 1), "num_samples": int(total)}


def train_local_scaffold(
    model: ResidualAdapterHead,
    x: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    *,
    learning_rate: float,
    weight_decay: float,
    batch_size: int,
    local_epochs: int,
    server_control: Mapping[str, torch.Tensor],
    client_control: Mapping[str, torch.Tensor],
    seed: int,
) -> dict[str, float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    total = 0
    steps = 0
    loss_sum = 0.0
    for epoch in range(int(local_epochs)):
        for xb, yb in make_loader(
            x, y, indices, batch_size, int(seed) + epoch, shuffle=True
        ):
            output = model(xb)
            loss = criterion(output["logits"], yb)
            model.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            with torch.no_grad():
                for name, parameter in model.named_parameters():
                    if parameter.grad is None:
                        continue
                    corrected = (
                        parameter.grad.detach().cpu()
                        + server_control[name]
                        - client_control[name]
                    )
                    if float(weight_decay) != 0.0:
                        corrected = corrected + float(weight_decay) * parameter.detach().cpu()
                    parameter -= float(learning_rate) * corrected.to(parameter.device)
            total += int(len(yb))
            steps += 1
            loss_sum += float(loss.detach().item()) * int(len(yb))
    return {
        "loss": loss_sum / max(total, 1),
        "num_samples": int(total),
        "num_steps": int(steps),
    }


@torch.no_grad()
def predict_model(
    model: ResidualAdapterHead, x: np.ndarray, batch_size: int
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probabilities = []
    features = []
    indices = np.arange(len(x), dtype=int)
    dummy = np.zeros(len(x), dtype=int)
    for xb, _ in make_loader(x, dummy, indices, batch_size, 0, shuffle=False):
        output = model(xb)
        probabilities.append(torch.softmax(output["logits"], dim=1).cpu().numpy())
        features.append(output["feature"].cpu().numpy())
    return (
        probability_normalize(np.concatenate(probabilities, axis=0)),
        np.concatenate(features, axis=0).astype(np.float32),
    )


def predict_ensemble(
    models: Sequence[ResidualAdapterHead], x: np.ndarray, batch_size: int
) -> np.ndarray:
    member_probabilities = [predict_model(model, x, batch_size)[0] for model in models]
    return probability_normalize(np.mean(np.stack(member_probabilities, axis=0), axis=0))


def extract_prototypes(
    model: ResidualAdapterHead, x: np.ndarray, y: np.ndarray, batch_size: int
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    _, features = predict_model(model, x, batch_size)
    prototypes = np.zeros((NUM_PHASES, features.shape[1]), dtype=np.float32)
    counts = np.zeros(NUM_PHASES, dtype=np.int64)
    labels = np.asarray(y, dtype=int)
    for phase in range(NUM_PHASES):
        items = features[labels == phase]
        if len(items):
            prototypes[phase] = items.mean(axis=0).astype(np.float32)
            counts[phase] = int(len(items))
    return prototypes, counts, counts > 0


def aggregate_prototypes(
    local_prototypes: Sequence[np.ndarray], local_counts: Sequence[np.ndarray]
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    total_counts = np.sum(np.stack(local_counts, axis=0), axis=0).astype(np.int64)
    global_prototypes = np.zeros_like(local_prototypes[0], dtype=np.float32)
    for phase in range(NUM_PHASES):
        if total_counts[phase] <= 0:
            continue
        weighted = np.zeros(global_prototypes.shape[1], dtype=np.float64)
        for prototypes, counts in zip(local_prototypes, local_counts):
            weighted += np.asarray(prototypes[phase], dtype=np.float64) * float(counts[phase])
        global_prototypes[phase] = (weighted / float(total_counts[phase])).astype(np.float32)
    return global_prototypes, total_counts, total_counts > 0


def prototype_probabilities_from_features(
    features: np.ndarray, prototypes: np.ndarray, valid: np.ndarray
) -> np.ndarray:
    distances = np.mean(
        (np.asarray(features)[:, None, :] - np.asarray(prototypes)[None, :, :]) ** 2,
        axis=2,
    )
    distances[:, ~np.asarray(valid, dtype=bool)] = 1e9
    return probability_normalize(np.exp(-(distances - distances.min(axis=1, keepdims=True))))


def prototype_ensemble_probabilities(
    models: Sequence[ResidualAdapterHead],
    prototypes: np.ndarray,
    valid: np.ndarray,
    x: np.ndarray,
    batch_size: int,
) -> np.ndarray:
    members = []
    for model in models:
        _, features = predict_model(model, x, batch_size)
        members.append(prototype_probabilities_from_features(features, prototypes, valid))
    return probability_normalize(np.mean(np.stack(members, axis=0), axis=0))

## 10. FedTGP Trainable Global-Prototype Server

FedTGP receives one uploaded prototype for every client/class pair present in a round. A persistent server module generates one global prototype per phase. The server objective uses Euclidean distances and the released nearest-gap adaptive margin; client learning uses cross-entropy plus MSE to the generated global prototype.

In [ ]:
class TrainableGlobalPrototypes(nn.Module):
    def __init__(self, num_classes: int, feature_dim: int) -> None:
        super().__init__()
        self.num_classes = int(num_classes)
        self.feature_dim = int(feature_dim)
        self.embeddings = nn.Embedding(self.num_classes, self.feature_dim)
        self.middle = nn.Sequential(nn.Linear(self.feature_dim, self.feature_dim), nn.ReLU())
        self.fc = nn.Linear(self.feature_dim, self.feature_dim)

    def forward(self, class_ids: torch.Tensor | Sequence[int] | int) -> torch.Tensor:
        ids = torch.as_tensor(class_ids, dtype=torch.long, device=self.embeddings.weight.device)
        return self.fc(self.middle(self.embeddings(ids)))


@dataclass(frozen=True)
class FedTGPServerResult:
    global_prototypes: np.ndarray
    valid: np.ndarray
    class_counts: np.ndarray
    adaptive_margin: float
    initial_loss: float
    best_loss: float
    epochs_run: int
    uploaded_bytes: int
    downloaded_bytes: int


def _uploaded_prototype_dataset(
    local_prototypes: Sequence[np.ndarray], local_counts: Sequence[np.ndarray]
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    vectors = []
    labels = []
    aggregate_counts = np.zeros(NUM_PHASES, dtype=np.int64)
    for prototypes, counts in zip(local_prototypes, local_counts):
        aggregate_counts += np.asarray(counts, dtype=np.int64)
        for phase in range(NUM_PHASES):
            if int(counts[phase]) > 0:
                vectors.append(np.asarray(prototypes[phase], dtype=np.float32))
                labels.append(int(phase))
    if not vectors:
        raise RuntimeError("FedTGP received no client prototypes.")
    return np.stack(vectors), np.asarray(labels, dtype=np.int64), aggregate_counts


def _adaptive_nearest_gap_margin(
    uploaded: np.ndarray, labels: np.ndarray, threshold: float
) -> float:
    centers = np.zeros((NUM_PHASES, uploaded.shape[1]), dtype=np.float32)
    valid = np.zeros(NUM_PHASES, dtype=bool)
    for phase in range(NUM_PHASES):
        items = uploaded[labels == phase]
        if len(items):
            centers[phase] = items.mean(axis=0)
            valid[phase] = True
    present = np.where(valid)[0]
    if len(present) < 2:
        return 0.0
    nearest = []
    for phase in present:
        distances = [
            float(np.linalg.norm(centers[phase] - centers[other]))
            for other in present
            if other != phase
        ]
        nearest.append(min(distances))
    return float(min(max(nearest), float(threshold)))


def update_fedtgp_server(
    server: TrainableGlobalPrototypes,
    local_prototypes: Sequence[np.ndarray],
    local_counts: Sequence[np.ndarray],
    *,
    server_epochs: int,
    learning_rate: float,
    batch_size: int,
    margin_threshold: float,
    seed: int,
) -> FedTGPServerResult:
    uploaded, labels, aggregate_counts = _uploaded_prototype_dataset(
        local_prototypes, local_counts
    )
    margin = _adaptive_nearest_gap_margin(uploaded, labels, margin_threshold)
    x = torch.tensor(uploaded, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)
    all_classes = torch.arange(NUM_PHASES, dtype=torch.long)
    optimizer = torch.optim.SGD(server.parameters(), lr=float(learning_rate))
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    effective_batch = min(max(int(batch_size), 1), len(x))

    def full_loss() -> torch.Tensor:
        generated = server(all_classes)
        distances = torch.cdist(x, generated, p=2)
        distances = distances + F.one_hot(y, num_classes=NUM_PHASES).float() * float(margin)
        return F.cross_entropy(-distances, y)

    server.eval()
    with torch.no_grad():
        initial_loss = float(full_loss().item())
    best_loss = initial_loss
    best_state = copy.deepcopy(server.state_dict())

    for epoch in range(int(server_epochs)):
        server.train()
        order = torch.randperm(len(x), generator=generator)
        for start in range(0, len(x), effective_batch):
            indices = order[start : start + effective_batch]
            generated = server(all_classes)
            distances = torch.cdist(x[indices], generated, p=2)
            distances = distances + F.one_hot(
                y[indices], num_classes=NUM_PHASES
            ).float() * float(margin)
            loss = F.cross_entropy(-distances, y[indices])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        server.eval()
        with torch.no_grad():
            value = float(full_loss().item())
        if value < best_loss:
            best_loss = value
            best_state = copy.deepcopy(server.state_dict())

    server.load_state_dict(best_state, strict=True)
    server.eval()
    with torch.no_grad():
        global_prototypes = server(all_classes).cpu().numpy().astype(np.float32)

    valid = aggregate_counts > 0
    uploaded_bytes = int(uploaded.nbytes + labels.nbytes)
    downloaded_bytes = int(global_prototypes.nbytes + np.arange(NUM_PHASES, dtype=np.int64).nbytes)
    return FedTGPServerResult(
        global_prototypes=global_prototypes,
        valid=valid,
        class_counts=aggregate_counts,
        adaptive_margin=float(margin),
        initial_loss=float(initial_loss),
        best_loss=float(best_loss),
        epochs_run=int(server_epochs),
        uploaded_bytes=uploaded_bytes,
        downloaded_bytes=downloaded_bytes,
    )

## 11. Protocol Implementations

Each function below returns remote probability distributions before evaluator labels are loaded. Source-validation Macro F1 selects the round and, when enabled, the candidate setting.

In [ ]:
@dataclass
class MethodRun:
    method: str
    candidate: dict[str, Any]
    best_val_macro_f1: float
    best_val_accuracy: float
    best_round: int
    remote_probabilities: np.ndarray
    training_bytes: int
    deployment_bytes: int
    history: pd.DataFrame


def source_validation_metrics(
    probabilities: np.ndarray, labels: np.ndarray
) -> dict[str, float]:
    return classification_metrics(labels, probabilities)


def train_fedavg(
    cfg: BenchmarkConfig,
    bundle: dict[str, Any],
    initial_state: Mapping[str, torch.Tensor],
    client_indices: list[np.ndarray],
    val_indices: np.ndarray,
    candidate: dict[str, Any],
    seed: int,
) -> MethodRun:
    x = bundle["source_seq_features"]
    labels = bundle["source_records"].sort_values("dataset_index")["true_phase_id"].to_numpy(dtype=int)
    global_state = copy.deepcopy(dict(initial_state))
    best_state = None
    best_macro = best_accuracy = -np.inf
    best_round = -1
    history = []
    training_bytes = 0

    for round_index in range(1, cfg.rounds + 1):
        local_states = []
        local_weights = []
        losses = []
        for client_index, indices in enumerate(client_indices):
            model = build_adapter(global_state, x.shape[1], cfg.bottleneck_dim)
            info = train_local_epochs(
                model,
                x,
                labels,
                indices,
                learning_rate=float(candidate["learning_rate"]),
                momentum=float(candidate.get("momentum", 0.0)),
                weight_decay=cfg.weight_decay,
                batch_size=int(candidate.get("batch_size", cfg.batch_size)),
                local_epochs=int(candidate.get("local_epochs", 1)),
                seed=seed * 100000 + round_index * 100 + client_index,
            )
            local_state = copy.deepcopy(model.state_dict())
            local_states.append(local_state)
            local_weights.append(float(len(indices)))
            losses.append(float(info["loss"]))
            training_bytes += logical_nbytes(global_state) + logical_nbytes(local_state)

        global_state = average_states(local_states, local_weights)
        validation_model = build_adapter(global_state, x.shape[1], cfg.bottleneck_dim)
        validation_probabilities = predict_model(
            validation_model, x[val_indices], cfg.inference_batch_size
        )[0]
        metrics = source_validation_metrics(validation_probabilities, labels[val_indices])
        history.append({
            "round": int(round_index),
            "train_loss": float(np.mean(losses)),
            "val_macro_f1": float(metrics["macro_f1"]),
            "val_accuracy": float(metrics["accuracy"]),
        })
        if better(metrics, best_macro, best_accuracy):
            best_macro = float(metrics["macro_f1"])
            best_accuracy = float(metrics["accuracy"])
            best_round = int(round_index)
            best_state = copy.deepcopy(global_state)

    if best_state is None:
        raise RuntimeError("FedAvg did not produce a validation-selected state.")
    best_model = build_adapter(best_state, x.shape[1], cfg.bottleneck_dim)
    remote = predict_model(best_model, bundle["remote_seq_features"], cfg.inference_batch_size)[0]
    return MethodRun(
        "fedavg_adapter",
        candidate,
        best_macro,
        best_accuracy,
        best_round,
        remote,
        int(training_bytes),
        int(state_logical_nbytes(best_state)),
        pd.DataFrame(history),
    )


def train_scaffold(
    cfg: BenchmarkConfig,
    bundle: dict[str, Any],
    initial_state: Mapping[str, torch.Tensor],
    client_indices: list[np.ndarray],
    val_indices: np.ndarray,
    candidate: dict[str, Any],
    seed: int,
) -> MethodRun:
    x = bundle["source_seq_features"]
    labels = bundle["source_records"].sort_values("dataset_index")["true_phase_id"].to_numpy(dtype=int)
    global_state = copy.deepcopy(dict(initial_state))
    server_control = zero_state_like(global_state)
    client_controls = [zero_state_like(global_state) for _ in client_indices]
    best_state = None
    best_macro = best_accuracy = -np.inf
    best_round = -1
    history = []
    training_bytes = 0
    learning_rate = float(candidate["learning_rate"])

    for round_index in range(1, cfg.rounds + 1):
        local_states = []
        control_deltas = []
        losses = []
        for client_index, indices in enumerate(client_indices):
            model = build_adapter(global_state, x.shape[1], cfg.bottleneck_dim)
            old_control = copy.deepcopy(client_controls[client_index])
            info = train_local_scaffold(
                model,
                x,
                labels,
                indices,
                learning_rate=learning_rate,
                weight_decay=cfg.weight_decay,
                batch_size=int(candidate.get("batch_size", cfg.batch_size)),
                local_epochs=int(candidate.get("local_epochs", 1)),
                server_control=server_control,
                client_control=old_control,
                seed=seed * 100000 + round_index * 100 + client_index,
            )
            local_state = copy.deepcopy(model.state_dict())
            steps = max(int(info["num_steps"]), 1)
            new_control = {
                key: old_control[key]
                - server_control[key]
                + (global_state[key] - local_state[key]) / (float(steps) * learning_rate)
                for key in global_state
            }
            control_delta = subtract_states(new_control, old_control)
            client_controls[client_index] = new_control
            local_states.append(local_state)
            control_deltas.append(control_delta)
            losses.append(float(info["loss"]))
            training_bytes += logical_nbytes({
                "global_state_down": global_state,
                "server_control_down": server_control,
                "client_state_up": local_state,
                "client_control_delta_up": control_delta,
            })

        global_state = average_states(local_states)
        server_control = add_states(server_control, average_states(control_deltas))
        validation_model = build_adapter(global_state, x.shape[1], cfg.bottleneck_dim)
        validation_probabilities = predict_model(
            validation_model, x[val_indices], cfg.inference_batch_size
        )[0]
        metrics = source_validation_metrics(validation_probabilities, labels[val_indices])
        history.append({
            "round": int(round_index),
            "train_loss": float(np.mean(losses)),
            "val_macro_f1": float(metrics["macro_f1"]),
            "val_accuracy": float(metrics["accuracy"]),
        })
        if better(metrics, best_macro, best_accuracy):
            best_macro = float(metrics["macro_f1"])
            best_accuracy = float(metrics["accuracy"])
            best_round = int(round_index)
            best_state = copy.deepcopy(global_state)

    if best_state is None:
        raise RuntimeError("SCAFFOLD did not produce a validation-selected state.")
    best_model = build_adapter(best_state, x.shape[1], cfg.bottleneck_dim)
    remote = predict_model(best_model, bundle["remote_seq_features"], cfg.inference_batch_size)[0]
    return MethodRun(
        "scaffold_adapter",
        candidate,
        best_macro,
        best_accuracy,
        best_round,
        remote,
        int(training_bytes),
        int(state_logical_nbytes(best_state)),
        pd.DataFrame(history),
    )

In [ ]:
def train_fedproto(
    cfg: BenchmarkConfig,
    bundle: dict[str, Any],
    initial_state: Mapping[str, torch.Tensor],
    client_indices: list[np.ndarray],
    val_indices: np.ndarray,
    candidate: dict[str, Any],
    seed: int,
) -> MethodRun:
    x = bundle["source_seq_features"]
    labels = bundle["source_records"].sort_values("dataset_index")["true_phase_id"].to_numpy(dtype=int)
    models = [build_adapter(initial_state, x.shape[1], cfg.bottleneck_dim) for _ in client_indices]
    global_prototypes = None
    valid = None
    counts = None
    best_states = best_prototypes = best_valid = best_counts = None
    best_macro = best_accuracy = -np.inf
    best_round = -1
    history = []
    training_bytes = 0

    for round_index in range(1, cfg.rounds + 1):
        losses = []
        for client_index, indices in enumerate(client_indices):
            info = train_local_epochs(
                models[client_index],
                x,
                labels,
                indices,
                learning_rate=float(candidate["learning_rate"]),
                momentum=float(candidate.get("momentum", 0.5)),
                weight_decay=cfg.weight_decay,
                batch_size=int(candidate.get("batch_size", cfg.batch_size)),
                local_epochs=int(candidate.get("local_epochs", 1)),
                seed=seed * 100000 + round_index * 100 + client_index,
                prototypes=global_prototypes,
                prototype_valid=valid,
                prototype_lambda=float(candidate["prototype_lambda"])
                if global_prototypes is not None
                else 0.0,
            )
            losses.append(float(info["loss"]))

        local_prototypes = []
        local_counts = []
        for client_index, indices in enumerate(client_indices):
            prototypes, phase_counts, _ = extract_prototypes(
                models[client_index], x[indices], labels[indices], cfg.inference_batch_size
            )
            local_prototypes.append(prototypes)
            local_counts.append(phase_counts)
            training_bytes += logical_nbytes({"prototypes": prototypes, "counts": phase_counts})

        global_prototypes, counts, valid = aggregate_prototypes(local_prototypes, local_counts)
        download_bundle = {"prototypes": global_prototypes, "counts": counts}
        training_bytes += len(client_indices) * logical_nbytes(download_bundle)

        validation_probabilities = prototype_ensemble_probabilities(
            models,
            global_prototypes,
            valid,
            x[val_indices],
            cfg.inference_batch_size,
        )
        metrics = source_validation_metrics(validation_probabilities, labels[val_indices])
        history.append({
            "round": int(round_index),
            "train_loss": float(np.mean(losses)),
            "val_macro_f1": float(metrics["macro_f1"]),
            "val_accuracy": float(metrics["accuracy"]),
        })
        if better(metrics, best_macro, best_accuracy):
            best_macro = float(metrics["macro_f1"])
            best_accuracy = float(metrics["accuracy"])
            best_round = int(round_index)
            best_states = [copy.deepcopy(model.state_dict()) for model in models]
            best_prototypes = np.asarray(global_prototypes, dtype=np.float32).copy()
            best_valid = np.asarray(valid, dtype=bool).copy()
            best_counts = np.asarray(counts, dtype=np.int64).copy()

    if best_states is None or best_prototypes is None or best_valid is None:
        raise RuntimeError("FedProto did not produce a validation-selected state.")
    best_models = [build_adapter(state, x.shape[1], cfg.bottleneck_dim) for state in best_states]
    remote = prototype_ensemble_probabilities(
        best_models,
        best_prototypes,
        best_valid,
        bundle["remote_seq_features"],
        cfg.inference_batch_size,
    )
    deployment_bytes = sum(state_logical_nbytes(state) for state in best_states) + logical_nbytes(
        {"prototypes": best_prototypes, "valid": best_valid, "counts": best_counts}
    )
    return MethodRun(
        "fedproto_adapter",
        candidate,
        best_macro,
        best_accuracy,
        best_round,
        remote,
        int(training_bytes),
        int(deployment_bytes),
        pd.DataFrame(history),
    )


def train_fedtgp(
    cfg: BenchmarkConfig,
    bundle: dict[str, Any],
    initial_state: Mapping[str, torch.Tensor],
    client_indices: list[np.ndarray],
    val_indices: np.ndarray,
    candidate: dict[str, Any],
    seed: int,
) -> MethodRun:
    x = bundle["source_seq_features"]
    labels = bundle["source_records"].sort_values("dataset_index")["true_phase_id"].to_numpy(dtype=int)
    models = [build_adapter(initial_state, x.shape[1], cfg.bottleneck_dim) for _ in client_indices]
    torch.manual_seed(int(seed) * 1000 + int(candidate.get("candidate_index", 0)))
    server = TrainableGlobalPrototypes(NUM_PHASES, x.shape[1])
    global_prototypes = None
    valid = None
    best_states = best_prototypes = best_valid = None
    best_macro = best_accuracy = -np.inf
    best_round = -1
    history = []
    training_bytes = 0

    for round_index in range(1, cfg.rounds + 1):
        losses = []
        for client_index, indices in enumerate(client_indices):
            info = train_local_epochs(
                models[client_index],
                x,
                labels,
                indices,
                learning_rate=float(candidate["learning_rate"]),
                momentum=0.0,
                weight_decay=cfg.weight_decay,
                batch_size=int(candidate.get("batch_size", cfg.batch_size)),
                local_epochs=int(candidate.get("local_epochs", 1)),
                seed=seed * 100000 + round_index * 100 + client_index,
                prototypes=global_prototypes,
                prototype_valid=valid,
                prototype_lambda=float(candidate["prototype_lambda"])
                if global_prototypes is not None
                else 0.0,
            )
            losses.append(float(info["loss"]))

        local_prototypes = []
        local_counts = []
        for client_index, indices in enumerate(client_indices):
            prototypes, counts, _ = extract_prototypes(
                models[client_index], x[indices], labels[indices], cfg.inference_batch_size
            )
            local_prototypes.append(prototypes)
            local_counts.append(counts)

        server_result = update_fedtgp_server(
            server,
            local_prototypes,
            local_counts,
            server_epochs=int(candidate.get("server_epochs", cfg.fedtgp_server_epochs)),
            learning_rate=float(candidate.get("server_learning_rate", candidate["learning_rate"])),
            batch_size=int(candidate.get("server_batch_size", cfg.fedtgp_server_batch_size)),
            margin_threshold=cfg.fedtgp_margin_threshold,
            seed=seed * 1000 + round_index,
        )
        global_prototypes = server_result.global_prototypes
        valid = server_result.valid
        training_bytes += int(
            server_result.uploaded_bytes + len(client_indices) * server_result.downloaded_bytes
        )

        validation_probabilities = prototype_ensemble_probabilities(
            models,
            global_prototypes,
            valid,
            x[val_indices],
            cfg.inference_batch_size,
        )
        metrics = source_validation_metrics(validation_probabilities, labels[val_indices])
        history.append({
            "round": int(round_index),
            "train_loss": float(np.mean(losses)),
            "adaptive_margin": float(server_result.adaptive_margin),
            "server_loss_initial": float(server_result.initial_loss),
            "server_loss_best": float(server_result.best_loss),
            "val_macro_f1": float(metrics["macro_f1"]),
            "val_accuracy": float(metrics["accuracy"]),
        })
        if better(metrics, best_macro, best_accuracy):
            best_macro = float(metrics["macro_f1"])
            best_accuracy = float(metrics["accuracy"])
            best_round = int(round_index)
            best_states = [copy.deepcopy(model.state_dict()) for model in models]
            best_prototypes = np.asarray(global_prototypes, dtype=np.float32).copy()
            best_valid = np.asarray(valid, dtype=bool).copy()

    if best_states is None or best_prototypes is None or best_valid is None:
        raise RuntimeError("FedTGP did not produce a validation-selected state.")
    best_models = [build_adapter(state, x.shape[1], cfg.bottleneck_dim) for state in best_states]
    remote = prototype_ensemble_probabilities(
        best_models,
        best_prototypes,
        best_valid,
        bundle["remote_seq_features"],
        cfg.inference_batch_size,
    )
    deployment_bytes = sum(state_logical_nbytes(state) for state in best_states) + logical_nbytes(
        {"trainable_global_prototypes": best_prototypes, "valid": best_valid}
    )
    return MethodRun(
        "fedtgp_adapter",
        candidate,
        best_macro,
        best_accuracy,
        best_round,
        remote,
        int(training_bytes),
        int(deployment_bytes),
        pd.DataFrame(history),
    )

In [ ]:
def ring_mixing_matrix(num_clients: int) -> np.ndarray:
    if int(num_clients) < 3:
        raise ValueError("D-PSGD ring gossip requires at least three clients.")
    matrix = np.zeros((num_clients, num_clients), dtype=np.float64)
    for client in range(num_clients):
        matrix[client, client] = 1.0 / 3.0
        matrix[client, (client - 1) % num_clients] = 1.0 / 3.0
        matrix[client, (client + 1) % num_clients] = 1.0 / 3.0
    if not (
        np.allclose(matrix.sum(axis=0), 1.0)
        and np.allclose(matrix.sum(axis=1), 1.0)
        and np.allclose(matrix, matrix.T)
    ):
        raise RuntimeError("Invalid D-PSGD mixing matrix.")
    return matrix


def train_dpsgd(
    cfg: BenchmarkConfig,
    bundle: dict[str, Any],
    initial_state: Mapping[str, torch.Tensor],
    client_indices: list[np.ndarray],
    val_indices: np.ndarray,
    candidate: dict[str, Any],
    seed: int,
) -> MethodRun:
    x = bundle["source_seq_features"]
    labels = bundle["source_records"].sort_values("dataset_index")["true_phase_id"].to_numpy(dtype=int)
    mixing = ring_mixing_matrix(len(client_indices))
    learning_rate = float(candidate["learning_rate"])
    batch_size = int(candidate.get("batch_size", cfg.batch_size))
    local_epochs = int(candidate.get("local_epochs", 1))
    max_batches = max(int(math.ceil(len(indices) / float(batch_size))) for indices in client_indices)
    steps_per_round = max_batches * local_epochs
    total_steps = cfg.rounds * steps_per_round
    models = [build_adapter(initial_state, x.shape[1], cfg.bottleneck_dim) for _ in client_indices]
    criterion = nn.CrossEntropyLoss()
    rng = np.random.default_rng(int(seed))
    orders = []
    positions = []
    for indices in client_indices:
        order = np.asarray(indices, dtype=int).copy()
        rng.shuffle(order)
        orders.append(order)
        positions.append(0)

    def next_batch(client_index: int) -> tuple[torch.Tensor, torch.Tensor]:
        order = orders[client_index]
        position = positions[client_index]
        if position + batch_size > len(order):
            rng.shuffle(order)
            position = 0
        selected = order[position : min(position + batch_size, len(order))]
        positions[client_index] = position + len(selected)
        return (
            torch.tensor(x[selected], dtype=torch.float32),
            torch.tensor(labels[selected], dtype=torch.long),
        )

    best_states = None
    best_macro = best_accuracy = -np.inf
    best_round = -1
    history = []
    state_bytes = state_logical_nbytes(models[0].state_dict())
    training_bytes = 0
    recent_losses = []

    for step in range(1, total_steps + 1):
        current_states = [copy.deepcopy(model.state_dict()) for model in models]
        gradients = []
        for client_index, model in enumerate(models):
            xb, yb = next_batch(client_index)
            model.zero_grad(set_to_none=True)
            output = model(xb)
            loss = criterion(output["logits"], yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            gradient_map = {}
            for name, parameter in model.named_parameters():
                gradient = (
                    torch.zeros_like(parameter.detach().cpu())
                    if parameter.grad is None
                    else parameter.grad.detach().cpu().clone()
                )
                if cfg.weight_decay != 0.0:
                    gradient = gradient + cfg.weight_decay * parameter.detach().cpu()
                gradient_map[name] = gradient
            gradients.append(gradient_map)
            recent_losses.append(float(loss.detach().item()))

        mixed_states = [
            average_states(current_states, mixing[client].tolist())
            for client in range(len(models))
        ]
        for client_index, model in enumerate(models):
            next_state = {
                key: mixed_states[client_index][key] - learning_rate * gradients[client_index][key]
                for key in mixed_states[client_index]
            }
            model.load_state_dict(next_state, strict=True)

        training_bytes += int(2 * len(models) * state_bytes)

        if step % steps_per_round == 0:
            round_index = step // steps_per_round
            validation_probabilities = predict_ensemble(
                models, x[val_indices], cfg.inference_batch_size
            )
            metrics = source_validation_metrics(validation_probabilities, labels[val_indices])
            history.append({
                "round": int(round_index),
                "step": int(step),
                "train_loss": float(np.mean(recent_losses)),
                "val_macro_f1": float(metrics["macro_f1"]),
                "val_accuracy": float(metrics["accuracy"]),
            })
            recent_losses = []
            if better(metrics, best_macro, best_accuracy):
                best_macro = float(metrics["macro_f1"])
                best_accuracy = float(metrics["accuracy"])
                best_round = int(round_index)
                best_states = [copy.deepcopy(model.state_dict()) for model in models]

    if best_states is None:
        raise RuntimeError("D-PSGD did not produce validation-selected peer states.")
    best_models = [build_adapter(state, x.shape[1], cfg.bottleneck_dim) for state in best_states]
    remote = predict_ensemble(best_models, bundle["remote_seq_features"], cfg.inference_batch_size)
    deployment_bytes = sum(state_logical_nbytes(state) for state in best_states)
    return MethodRun(
        "dpsgd_ring_gossip_adapter",
        candidate,
        best_macro,
        best_accuracy,
        best_round,
        remote,
        int(training_bytes),
        int(deployment_bytes),
        pd.DataFrame(history),
    )


def load_dmet_hub(path: Path) -> dict[str, Any]:
    raw = torch.load(path, map_location="cpu", weights_only=False)
    prototypes = l2_normalize_np(np.asarray(raw["embeddings"], dtype=np.float32))
    if prototypes.shape != (NUM_PHASES, 128):
        raise RuntimeError(f"Expected an [8, 128] hub prototype matrix, found {prototypes.shape}")
    calibration = raw.get("calibration", {}) or {}
    dirichlet = calibration.get("dirichlet_calibrator")
    conformal = calibration.get("conformal_calibrator")
    if dirichlet is None or conformal is None or "qhat" not in conformal:
        raise RuntimeError("The DMET hub lacks Dirichlet calibration or conformal qhat.")
    return {"prototypes": prototypes, "dirichlet": dirichlet, "conformal": conformal}


def dirichlet_hub_probabilities(
    queries: np.ndarray, prototypes: np.ndarray, calibrator: Mapping[str, Any]
) -> np.ndarray:
    raw = np.clip(
        softmax_np(l2_normalize_np(queries) @ l2_normalize_np(prototypes).T, axis=1),
        EPS,
        1.0,
    )
    log_raw = np.log(raw)
    coefficients = np.asarray(calibrator["coef"], dtype=np.float64)
    intercept = np.asarray(calibrator["intercept"], dtype=np.float64)
    classes = np.asarray(calibrator["classes"], dtype=int)
    observed = softmax_np(log_raw @ coefficients.T + intercept.reshape(1, -1), axis=1)
    probabilities = np.zeros((len(queries), NUM_PHASES), dtype=np.float64)
    for column, phase in enumerate(classes):
        probabilities[:, int(phase)] = observed[:, column]
    return probability_normalize(probabilities)


def conformal_restricted_targets(
    probabilities: np.ndarray, qhat: float
) -> tuple[np.ndarray, np.ndarray]:
    targets = []
    sizes = []
    for probability in probability_normalize(probabilities):
        order = np.argsort(-probability)
        cumulative = np.cumsum(probability[order])
        crossing = np.where(cumulative > float(qhat))[0]
        last = int(crossing[0]) if len(crossing) else len(order) - 1
        selected = order[: last + 1]
        target = np.zeros(NUM_PHASES, dtype=np.float64)
        target[selected] = probability[selected]
        target = probability_normalize(target)
        targets.append(target)
        sizes.append(int(len(selected)))
    return np.asarray(targets), np.asarray(sizes, dtype=int)


def dmet_direct_probabilities(cfg: BenchmarkConfig, bundle: dict[str, Any]) -> tuple[np.ndarray, int]:
    hub = load_dmet_hub(cfg.dmet_hub)
    evidence = dirichlet_hub_probabilities(
        bundle["remote_selected_motif_embeddings"],
        hub["prototypes"],
        hub["dirichlet"],
    )
    targets, set_sizes = conformal_restricted_targets(
        evidence, float(hub["conformal"]["qhat"])
python -m pip install -e    )
    weights = np.where(set_sizes == 1, 1.0, np.where(set_sizes <= 3, 0.5, 0.0))
    local = probability_normalize(bundle["remote_seq_probs"])
    final = probability_normalize((1.0 - weights[:, None]) * local + weights[:, None] * targets)
    # Current manuscript accounting: 128 float32 query values + eight float16 response values.
    communication_bytes = int(len(final) * (128 * 4 + NUM_PHASES * 2))
    return final, communication_bytes

## 12. Candidate Settings and Complete Benchmark Runner

Candidate selection never uses remote labels. In `compact_grid` mode, seed 42 selects the candidate with the highest source-validation Macro F1, using source-validation accuracy only as a tie-breaker. The selected candidate is then reused unchanged for seeds 43 and 44.

In [ ]:
RETAINED_CANDIDATES = {
    "fedavg_adapter": [
        {"candidate_index": 0, "learning_rate": 0.001, "local_epochs": 5, "batch_size": 64, "momentum": 0.0}
    ],
    "scaffold_adapter": [
        {"candidate_index": 0, "learning_rate": 0.001, "local_epochs": 5, "batch_size": 64}
    ],
    "fedproto_adapter": [
        {"candidate_index": 0, "learning_rate": 0.005, "prototype_lambda": 0.1, "momentum": 0.5, "local_epochs": 1, "batch_size": 64}
    ],
    "fedtgp_adapter": [
        {"candidate_index": 0, "learning_rate": 0.01, "prototype_lambda": 1.0, "local_epochs": 1, "batch_size": 64, "server_learning_rate": 0.01, "server_batch_size": 10}
    ],
    "dpsgd_ring_gossip_adapter": [
        {"candidate_index": 0, "learning_rate": 0.001, "local_epochs": 5, "batch_size": 64}
    ],
}

COMPACT_CANDIDATE_GRID = {
    "fedavg_adapter": [
        {"learning_rate": lr, "local_epochs": epochs, "batch_size": 64, "momentum": 0.0}
        for lr in (0.001, 0.005, 0.01)
        for epochs in (1, 5)
    ],
    "scaffold_adapter": [
        {"learning_rate": lr, "local_epochs": epochs, "batch_size": 64}
        for lr in (0.001, 0.005, 0.01)
        for epochs in (1, 5)
    ],
    "fedproto_adapter": [
        {"learning_rate": lr, "prototype_lambda": lam, "momentum": 0.5, "local_epochs": 1, "batch_size": 64}
        for lr in (0.005, 0.01)
        for lam in (0.1, 1.0)
    ],
    "fedtgp_adapter": [
        {"learning_rate": lr, "prototype_lambda": lam, "local_epochs": 1, "batch_size": 64, "server_learning_rate": lr, "server_batch_size": 10}
        for lr in (0.005, 0.01)
        for lam in (0.1, 1.0)
    ],
    "dpsgd_ring_gossip_adapter": [
        {"learning_rate": lr, "local_epochs": 5, "batch_size": 64}
        for lr in (0.001, 0.005, 0.01)
    ],
}
for method, candidates in COMPACT_CANDIDATE_GRID.items():
    for index, candidate in enumerate(candidates):
        candidate["candidate_index"] = int(index)


TRAINING_FUNCTIONS = {
    "fedavg_adapter": train_fedavg,
    "scaffold_adapter": train_scaffold,
    "fedproto_adapter": train_fedproto,
    "fedtgp_adapter": train_fedtgp,
    "dpsgd_ring_gossip_adapter": train_dpsgd,
}


def candidates_for_method(
    cfg: BenchmarkConfig,
    method: str,
    seed: int,
    locked_candidates: Mapping[str, dict[str, Any]],
) -> list[dict[str, Any]]:
    if seed != cfg.selection_seed:
        if method not in locked_candidates:
            raise RuntimeError(f"No candidate was locked for {method} on seed {cfg.selection_seed}.")
        return [copy.deepcopy(locked_candidates[method])]
    source = RETAINED_CANDIDATES if cfg.hyperparameter_mode == "retained" else COMPACT_CANDIDATE_GRID
    candidates = [copy.deepcopy(candidate) for candidate in source[method]]
    if cfg.hyperparameter_mode == "retained" and method == "fedtgp_adapter":
        candidates[0]["server_epochs"] = cfg.fedtgp_server_epochs
    elif method == "fedtgp_adapter":
        for candidate in candidates:
            candidate["server_epochs"] = cfg.fedtgp_server_epochs
    return candidates


@dataclass
class BenchmarkOutputs:
    config: BenchmarkConfig
    seed_metrics: pd.DataFrame
    communication: pd.DataFrame
    predictions: pd.DataFrame
    locked_candidates: dict[str, dict[str, Any]]
    partition_diagnostics: pd.DataFrame


def prediction_packet(
    records: pd.DataFrame, probabilities: np.ndarray, method: str, seed: int
) -> pd.DataFrame:
    ordered = records.sort_values("dataset_index").reset_index(drop=True)
    packet = ordered[["dataset_index", "case_id", "fragment_file"]].copy()
    packet.insert(0, "method", method)
    packet.insert(0, "seed", int(seed))
    for phase in range(NUM_PHASES):
        packet[f"prob_phase_{phase}"] = probabilities[:, phase]
    return packet


def run_full_comparative_benchmark() -> BenchmarkOutputs:
    cfg = build_config()
    os.environ["OMP_NUM_THREADS"] = str(cfg.cpu_threads)
    os.environ["MKL_NUM_THREADS"] = str(cfg.cpu_threads)
    torch.set_num_threads(cfg.cpu_threads)
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    cfg.run_dir.mkdir(parents=True, exist_ok=True)

    bundle = export_or_load_feature_bundle(cfg)
    source_records = bundle["source_records"].sort_values("dataset_index").reset_index(drop=True)
    remote_records = bundle["remote_records"].sort_values("dataset_index").reset_index(drop=True)
    train_indices = source_records[source_records["split"].astype(str) == "train"]["dataset_index"].to_numpy(dtype=int)
    validation_indices = source_records[source_records["split"].astype(str) == "validation"]["dataset_index"].to_numpy(dtype=int)

    locked_candidates: dict[str, dict[str, Any]] = {}
    prediction_packets = []
    communication_rows = []
    partition_rows = []

    for seed in cfg.seeds:
        print(f"\n================ SEED {seed} ================")
        set_seed(seed)
        partition = build_casewise_non_iid_partition(
            source_records,
            seed,
            num_clients=cfg.num_clients,
            search_trials=cfg.partition_search_trials,
            min_phases_per_client=cfg.partition_min_phases_per_client,
            min_clients_per_phase=cfg.partition_min_clients_per_phase,
            max_fragment_ratio=cfg.partition_max_fragment_ratio,
        )
        partition_frame = partition.diagnostics.copy()
        partition_frame.insert(0, "seed", int(seed))
        partition_rows.append(partition_frame)

        initial_state = create_initial_adapter_state(
            input_dim=bundle["source_seq_features"].shape[1],
            bottleneck_dim=cfg.bottleneck_dim,
            base_weight=bundle["base_fc_weight"],
            base_bias=bundle["base_fc_bias"],
            seed=seed,
        )

        source_probabilities = probability_normalize(bundle["remote_seq_probs"])
        prediction_packets.append(
            prediction_packet(remote_records, source_probabilities, "source_only", seed)
        )
        communication_rows.append({
            "seed": int(seed),
            "method": "source_only",
            "training_bytes": 0,
            "deployment_bytes": 0,
            "total_protocol_bytes": 0,
        })

        for method in (
            "fedavg_adapter",
            "scaffold_adapter",
            "fedproto_adapter",
            "fedtgp_adapter",
            "dpsgd_ring_gossip_adapter",
        ):
            candidates = candidates_for_method(cfg, method, seed, locked_candidates)
            candidate_runs = []
            for candidate in candidates:
                print(f"{DISPLAY_NAMES[method]} candidate {candidate['candidate_index']}: {candidate}")
                result = TRAINING_FUNCTIONS[method](
                    cfg,
                    bundle,
                    initial_state,
                    partition.client_indices,
                    validation_indices,
                    candidate,
                    seed,
                )
                candidate_runs.append(result)
            selected = max(
                candidate_runs,
                key=lambda result: (result.best_val_macro_f1, result.best_val_accuracy),
            )
            if seed == cfg.selection_seed:
                locked_candidates[method] = copy.deepcopy(selected.candidate)
            prediction_packets.append(
                prediction_packet(remote_records, selected.remote_probabilities, method, seed)
            )
            communication_rows.append({
                "seed": int(seed),
                "method": method,
                "training_bytes": int(selected.training_bytes),
                "deployment_bytes": int(selected.deployment_bytes),
                "total_protocol_bytes": int(selected.training_bytes + selected.deployment_bytes),
                "best_source_validation_macro_f1": float(selected.best_val_macro_f1),
                "best_round": int(selected.best_round),
                "candidate_json": json.dumps(selected.candidate, sort_keys=True),
            })

        dmet_probabilities, dmet_bytes = dmet_direct_probabilities(cfg, bundle)
        prediction_packets.append(
            prediction_packet(remote_records, dmet_probabilities, "dmet_direct", seed)
        )
        communication_rows.append({
            "seed": int(seed),
            "method": "dmet_direct",
            "training_bytes": 0,
            "deployment_bytes": 0,
            "query_bytes_per_query": 128 * 4,
            "response_bytes_per_query": NUM_PHASES * 2,
            "num_remote_queries": int(len(remote_records)),
            "total_protocol_bytes": int(dmet_bytes),
        })

    public_predictions = pd.concat(prediction_packets, ignore_index=True)
    # Remote labels are loaded only after every public method packet exists.
    remote_labels = pd.read_csv(bundle["remote_labels_path"])
    evaluated = public_predictions.merge(
        remote_labels[["dataset_index", "case_id", "fragment_file", "true_phase_id"]],
        on="dataset_index",
        suffixes=("", "_evaluator"),
        validate="many_to_one",
    )
    if not (evaluated["case_id"].astype(str) == evaluated["case_id_evaluator"].astype(str)).all():
        raise RuntimeError("Case mismatch while evaluator labels were attached.")
    if not (
        evaluated["fragment_file"].astype(str)
        == evaluated["fragment_file_evaluator"].astype(str)
    ).all():
        raise RuntimeError("Fragment mismatch while evaluator labels were attached.")
    evaluated = evaluated.drop(columns=["case_id_evaluator", "fragment_file_evaluator"])

    metric_rows = []
    probability_columns = [f"prob_phase_{phase}" for phase in range(NUM_PHASES)]
    for (seed, method), subset in evaluated.groupby(["seed", "method"], sort=False):
        metrics = classification_metrics(
            subset["true_phase_id"].to_numpy(dtype=int),
            subset[probability_columns].to_numpy(dtype=float),
        )
        metric_rows.append({
            "seed": int(seed),
            "method": str(method),
            "display_name": DISPLAY_NAMES[str(method)],
            "num_remote_fragments": int(len(subset)),
            **metrics,
        })

    return BenchmarkOutputs(
        config=cfg,
        seed_metrics=pd.DataFrame(metric_rows),
        communication=pd.DataFrame(communication_rows),
        predictions=evaluated,
        locked_candidates=locked_candidates,
        partition_diagnostics=pd.concat(partition_rows, ignore_index=True),
    )

## 13. Run the Complete Comparison

This call validates the five input artifacts, builds or reuses the internal frozen features, creates deterministic source-client partitions, runs all seven methods, and attaches remote evaluator labels only after the public prediction distributions have been produced.

In [ ]:
benchmark = run_full_comparative_benchmark()
print("Comparative methods completed.")

## 14. Comparative Table, Communication Data, and Seed-Level Figure

The table reports the mean remote Macro F1 and accuracy across seeds, Macro-F1 gain relative to the source-only reference, and the protocol payload. The figure presents each seed, the across-seed mean, the observed seed range, and the aligned communication footprint.

In [ ]:

# =========================================================
# COMPARATIVE OUTPUTS
# Figure styling matched to the manuscript version
# =========================================================

import json
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def build_comparative_summary(benchmark: BenchmarkOutputs) -> pd.DataFrame:
    metrics = benchmark.seed_metrics.copy()
    communication = benchmark.communication.copy()
    source = metrics[metrics["method"] == "source_only"].set_index("seed")

    rows = []
    for method in METHOD_ORDER:
        method_metrics = metrics[metrics["method"] == method].copy()
        method_comm = communication[communication["method"] == method].copy()

        gains = [
            float(row["macro_f1"])
            - float(source.loc[int(row["seed"]), "macro_f1"])
            for _, row in method_metrics.iterrows()
        ]

        rows.append(
            {
                "method": method,
                "display_name": DISPLAY_NAMES[method],
                "protocol": PROTOCOL_DESCRIPTIONS[method],
                "exchanged_information": EXCHANGED_INFORMATION[method],
                "macro_f1_percent": 100.0
                * float(method_metrics["macro_f1"].mean()),
                "macro_f1_sd_percent": (
                    100.0
                    * float(method_metrics["macro_f1"].std(ddof=1))
                    if len(method_metrics) > 1
                    else 0.0
                ),
                "accuracy_percent": 100.0
                * float(method_metrics["accuracy"].mean()),
                "macro_f1_gain_vs_source_pp": (
                    0.0
                    if method == "source_only"
                    else 100.0 * float(np.mean(gains))
                ),
                "communication_payload_mib": float(
                    method_comm["total_protocol_bytes"].mean()
                )
                / (1024.0**2),
            }
        )

    return pd.DataFrame(rows)


def latex_escape(text: str) -> str:
    return (
        str(text)
        .replace("&", r"\&")
        .replace("%", r"\%")
        .replace("_", r"\_")
    )


def write_comparative_latex(
    summary: pd.DataFrame,
    path: Path,
) -> None:
    lines = [
        r"\begin{tabular}{p{2.6cm}p{4.4cm}p{3.0cm}ccc}",
        r"\hline",
        (
            r"Method & Protocol & Exchanged information & "
            r"Macro F1 / Acc. (\%) & Gain (pp) & Payload (MiB) \\"
        ),
        r"\hline",
    ]

    for _, row in summary.iterrows():
        gain = (
            "Reference"
            if row["method"] == "source_only"
            else f"{row['macro_f1_gain_vs_source_pp']:+.2f}"
        )
        payload = (
            "--"
            if row["method"] == "source_only"
            else f"{row['communication_payload_mib']:.3f}"
        )

        lines.append(
            f"{latex_escape(row['display_name'])} & "
            f"{latex_escape(row['protocol'])} & "
            f"{latex_escape(row['exchanged_information'])} & "
            f"{row['macro_f1_percent']:.2f} / "
            f"{row['accuracy_percent']:.2f} & "
            f"{gain} & {payload}"
            + r" \\"
        )

    lines.extend([r"\hline", r"\end{tabular}"])
    path.write_text("\n".join(lines), encoding="utf-8")


def make_seed_payload_figure(
    seed_metrics: pd.DataFrame,
    communication: pd.DataFrame,
    output_stem: Path,
) -> pd.DataFrame:
    """
    Generate the manuscript-style two-panel figure:

    Panel A
        Seed-level remote Macro F1, across-seed means, and min--max ranges.

    Panel B
        Logical protocol communication payload on a logarithmic scale.

    All values are read from `seed_metrics` and `communication`.
    No method score or payload value is hard-coded.
    """

    figure_order = [
        "source_only",
        "dmet_direct",
        "fedtgp_adapter",
        "fedproto_adapter",
        "fedavg_adapter",
        "scaffold_adapter",
        "dpsgd_ring_gossip_adapter",
    ]

    method_labels = {
        "source_only": "Source-only\nref.",
        "dmet_direct": "DMET-\nSSTKG",
        "fedtgp_adapter": "FedTGP",
        "fedproto_adapter": "FedProto",
        "fedavg_adapter": "FedAvg",
        "scaffold_adapter": "SCAFFOLD",
        "dpsgd_ring_gossip_adapter": "D-PSGD",
    }

    method_colors = {
        "source_only": "#6B6B6B",
        "dmet_direct": "#C51B7D",
        "fedtgp_adapter": "#D95F02",
        "fedproto_adapter": "#7570B3",
        "fedavg_adapter": "#4D4D4D",
        "scaffold_adapter": "#1B9E77",
        "dpsgd_ring_gossip_adapter": "#8C564B",
    }

    seed_markers = {
        42: "o",
        43: "s",
        44: "^",
    }

    seed_colors = {
        42: "#1f77b4",
        43: "#d62728",
        44: "#2ca02c",
    }

    seed_x_offsets = {
        42: -0.16,
        43: 0.00,
        44: 0.16,
    }

    required_metric_columns = {
        "seed",
        "method",
        "macro_f1",
        "accuracy",
    }
    required_comm_columns = {
        "method",
        "total_protocol_bytes",
    }

    missing_metrics = required_metric_columns.difference(seed_metrics.columns)
    missing_comm = required_comm_columns.difference(communication.columns)

    if missing_metrics:
        raise ValueError(
            "seed_metrics is missing required columns: "
            + ", ".join(sorted(missing_metrics))
        )
    if missing_comm:
        raise ValueError(
            "communication is missing required columns: "
            + ", ".join(sorted(missing_comm))
        )

    available_methods = set(seed_metrics["method"].astype(str))
    missing_methods = [
        method for method in figure_order if method not in available_methods
    ]
    if missing_methods:
        raise ValueError(
            "Seed-level results are missing methods required by the figure: "
            + ", ".join(missing_methods)
        )

    plot_metrics = seed_metrics.copy()
    plot_metrics["macro_f1_percent"] = (
        100.0 * plot_metrics["macro_f1"].astype(float)
    )
    plot_metrics["accuracy_percent"] = (
        100.0 * plot_metrics["accuracy"].astype(float)
    )

    payload_df = (
        communication.groupby("method", as_index=False)[
            "total_protocol_bytes"
        ]
        .mean()
        .rename(
            columns={
                "total_protocol_bytes": "mean_total_protocol_bytes"
            }
        )
    )
    payload_df["total_protocol_mib"] = (
        payload_df["mean_total_protocol_bytes"] / (1024.0**2)
    )

    summary_df = (
        plot_metrics.groupby("method", as_index=False)
        .agg(
            mean_macro_f1=("macro_f1_percent", "mean"),
            min_macro_f1=("macro_f1_percent", "min"),
            max_macro_f1=("macro_f1_percent", "max"),
            sd_macro_f1=("macro_f1_percent", "std"),
            mean_accuracy=("accuracy_percent", "mean"),
        )
        .merge(
            payload_df[
                [
                    "method",
                    "total_protocol_mib",
                ]
            ],
            on="method",
            how="left",
            validate="one_to_one",
        )
    )

    summary_df["seed_range_pp"] = (
        summary_df["max_macro_f1"]
        - summary_df["min_macro_f1"]
    )
    summary_df["order"] = summary_df["method"].map(
        {method: index for index, method in enumerate(figure_order)}
    )
    summary_df = summary_df.sort_values("order").reset_index(drop=True)

    source_macro = float(
        summary_df.loc[
            summary_df["method"] == "source_only",
            "mean_macro_f1",
        ].iloc[0]
    )
    dmet_payload = float(
        summary_df.loc[
            summary_df["method"] == "dmet_direct",
            "total_protocol_mib",
        ].iloc[0]
    )

    if not np.isfinite(dmet_payload) or dmet_payload <= 0:
        raise ValueError(
            "DMET-SSTKG payload must be positive for ratio calculation."
        )

    summary_df["payload_ratio_vs_dmet"] = np.where(
        summary_df["method"] == "source_only",
        np.nan,
        summary_df["total_protocol_mib"] / dmet_payload,
    )

    # Save the exact data used to draw the figure.
    summary_csv = output_stem.parent / (
        output_stem.name + "_summary.csv"
    )
    summary_df.to_csv(summary_csv, index=False)

    # -----------------------------------------------------
    # Publication styling
    # -----------------------------------------------------
    plt.rcParams.update(
        {
            "font.size": 11.5,
            "axes.titlesize": 13.0,
            "axes.labelsize": 12.5,
            "xtick.labelsize": 10.8,
            "ytick.labelsize": 10.8,
            "legend.fontsize": 9.4,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )

    fig = plt.figure(figsize=(10.8, 6.25))
    grid = fig.add_gridspec(
        2,
        1,
        height_ratios=[4.15, 1.25],
        hspace=0.12,
    )
    ax_top = fig.add_subplot(grid[0, 0])
    ax_bottom = fig.add_subplot(grid[1, 0], sharex=ax_top)

    x_positions = np.arange(len(figure_order))

    # Highlight source-only and proposed method columns.
    for position, method in enumerate(figure_order):
        if method == "source_only":
            shade_color = "#F2F2F2"
            shade_alpha = 0.90
        elif method == "dmet_direct":
            shade_color = "#FCEAF4"
            shade_alpha = 0.85
        else:
            continue

        for axis in (ax_top, ax_bottom):
            axis.axvspan(
                position - 0.5,
                position + 0.5,
                color=shade_color,
                alpha=shade_alpha,
                zorder=0,
            )

    # Source-only horizontal reference.
    ax_top.axhline(
        source_macro,
        color="0.37",
        linestyle="--",
        linewidth=1.10,
        alpha=0.82,
        zorder=1,
    )

    all_macro_values = plot_metrics["macro_f1_percent"].to_numpy(
        dtype=float
    )
    data_min = float(np.min(all_macro_values))
    data_max = float(np.max(all_macro_values))
    top_padding = max(0.35, 0.12 * (data_max - data_min))
    bottom_padding = max(0.25, 0.08 * (data_max - data_min))
    y_lower = np.floor((data_min - bottom_padding) * 4.0) / 4.0
    y_upper = np.ceil((data_max + top_padding) * 4.0) / 4.0

    ax_top.text(
        0.13,
        source_macro + 0.055,
        "Source-only reference",
        fontsize=10.2,
        fontweight="bold",
        color="0.32",
        ha="left",
        va="bottom",
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.88,
            "pad": 0.8,
        },
        zorder=20,
    )

    seed_values = sorted(
        int(seed) for seed in plot_metrics["seed"].unique()
    )

    # Extend marker definitions safely when another seed is present.
    fallback_markers = ["o", "s", "^", "P", "X", "v", "D"]
    fallback_offsets = np.linspace(-0.18, 0.18, max(len(seed_values), 1))

    for index, seed in enumerate(seed_values):
        seed_markers.setdefault(
            seed,
            fallback_markers[index % len(fallback_markers)],
        )
        seed_x_offsets.setdefault(seed, float(fallback_offsets[index]))

    for position, method in enumerate(figure_order):
        method_summary = summary_df[
            summary_df["method"] == method
        ].iloc[0]
        method_seed_rows = plot_metrics[
            plot_metrics["method"] == method
        ].sort_values("seed")

        color = method_colors[method]
        is_dmet = method == "dmet_direct"
        mean_value = float(method_summary["mean_macro_f1"])
        min_value = float(method_summary["min_macro_f1"])
        max_value = float(method_summary["max_macro_f1"])

        # Min--max range.
        if np.isclose(min_value, max_value):
            ax_top.plot(
                [position - 0.12, position + 0.12],
                [mean_value, mean_value],
                color=color,
                linewidth=2.20 if is_dmet else 1.65,
                alpha=0.70,
                zorder=4,
            )
        else:
            ax_top.plot(
                [position, position],
                [min_value, max_value],
                color=color,
                linewidth=2.20 if is_dmet else 1.65,
                alpha=0.70,
                zorder=4,
            )
            ax_top.plot(
                [position - 0.075, position + 0.075],
                [min_value, min_value],
                color=color,
                linewidth=1.45,
                alpha=0.70,
                zorder=4,
            )
            ax_top.plot(
                [position - 0.075, position + 0.075],
                [max_value, max_value],
                color=color,
                linewidth=1.45,
                alpha=0.70,
                zorder=4,
            )

        # Seed-level points.
        for _, seed_row in method_seed_rows.iterrows():
            seed = int(seed_row["seed"])
            seed_value = float(seed_row["macro_f1_percent"])

            if method == "source_only":
                face_color = "white"
            else:
                face_color = seed_colors.get(seed, color)

            ax_top.scatter(
                position + seed_x_offsets.get(seed, 0.0),
                seed_value,
                s=54 if is_dmet else 44,
                marker=seed_markers.get(seed, "o"),
                facecolors=face_color,
                edgecolors="black",
                linewidths=0.70,
                alpha=0.95,
                zorder=8,
            )

        # Across-seed mean.
        ax_top.scatter(
            position,
            mean_value,
            s=150 if is_dmet else 90,
            marker="D",
            facecolors=color,
            edgecolors="black",
            linewidths=1.05,
            alpha=0.98,
            zorder=11,
        )

        if is_dmet:
            ax_top.scatter(
                position,
                mean_value,
                s=235,
                marker="D",
                facecolors="none",
                edgecolors=color,
                linewidths=1.35,
                alpha=0.35,
                zorder=10,
            )

            dmet_is_highest = np.isclose(
                mean_value,
                float(summary_df["mean_macro_f1"].max()),
            )
            first_line = (
                "highest mean"
                if dmet_is_highest
                else "DMET-SSTKG mean"
            )
            annotation = (
                f"{first_line}\n"
                f"{float(method_summary['seed_range_pp']):.2f} pp "
                "seed range"
            )

            annotation_y = mean_value - max(
                0.45,
                0.16 * (y_upper - y_lower),
            )
            ax_top.text(
                position,
                annotation_y,
                annotation,
                fontsize=9.5,
                fontweight="bold",
                ha="center",
                va="top",
                color="0.10",
                bbox={
                    "facecolor": "white",
                    "edgecolor": "0.78",
                    "linewidth": 0.40,
                    "alpha": 0.90,
                    "boxstyle": "round,pad=0.18",
                },
                zorder=30,
            )

    ax_top.set_xlim(-0.55, len(figure_order) - 0.45)
    ax_top.set_ylim(y_lower, y_upper)
    ax_top.set_ylabel(
        "Remote Macro F1 (%)",
        fontsize=13.0,
        fontweight="bold",
        labelpad=8,
    )
    ax_top.set_title(
        "A  Seed-level performance across random seeds",
        loc="left",
        fontsize=13.2,
        fontweight="bold",
        pad=8,
    )
    ax_top.grid(
        True,
        axis="y",
        alpha=0.25,
        linewidth=0.70,
    )
    ax_top.grid(
        True,
        axis="x",
        alpha=0.08,
        linewidth=0.40,
    )
    ax_top.tick_params(axis="x", labelbottom=False)

    for spine in ax_top.spines.values():
        spine.set_linewidth(1.10)
        spine.set_color("black")
    for tick in ax_top.get_yticklabels():
        tick.set_fontweight("bold")

    legend_handles = [
        mlines.Line2D(
            [],
            [],
            color="white",
            marker=seed_markers[seed],
            markerfacecolor=seed_colors.get(seed, "white"),
            markeredgecolor="black",
            linestyle="None",
            markersize=7.5,
            label=f"Seed {seed}",
        )
        for seed in seed_values
    ]
    legend_handles.extend(
        [
            mlines.Line2D(
                [],
                [],
                color="black",
                marker="D",
                markerfacecolor="0.78",
                markeredgecolor="black",
                linestyle="None",
                markersize=8.0,
                label="Mean",
            ),
            mlines.Line2D(
                [],
                [],
                color="0.35",
                linewidth=1.6,
                marker="_",
                markersize=9.0,
                markeredgewidth=1.6,
                label="Min--max",
            ),
        ]
    )

    ax_top.legend(
        handles=legend_handles,
        loc="upper right",
        bbox_to_anchor=(0.995, 0.995),
        ncol=min(len(legend_handles), 5),
        frameon=True,
        facecolor="white",
        edgecolor="0.82",
        framealpha=0.88,
        fontsize=9.3,
        handlelength=1.45,
        columnspacing=1.00,
        borderpad=0.45,
    )

    # -----------------------------------------------------
    # Communication panel
    # -----------------------------------------------------
    payload_by_method = dict(
        zip(
            summary_df["method"],
            summary_df["total_protocol_mib"],
        )
    )

    positive_payloads = np.asarray(
        [
            float(payload_by_method[method])
            for method in figure_order
            if method != "source_only"
            and float(payload_by_method[method]) > 0
        ],
        dtype=float,
    )
    if positive_payloads.size == 0:
        raise ValueError(
            "No positive communication payload was found."
        )

    lower_comm_limit = min(0.42, float(positive_payloads.min()) / 1.25)
    upper_comm_limit = max(4000.0, float(positive_payloads.max()) * 1.55)
    line_start = max(lower_comm_limit * 1.30, 1e-6)

    ax_bottom.set_yscale("log")
    ax_bottom.set_ylim(lower_comm_limit, upper_comm_limit)
    ax_bottom.set_title(
        "B  Communication footprint aligned to the same methods",
        loc="left",
        fontsize=12.0,
        fontweight="bold",
        pad=5,
    )

    for position, method in enumerate(figure_order):
        color = method_colors[method]

        if method == "source_only":
            source_line_y = line_start
            ax_bottom.plot(
                [position - 0.21, position + 0.21],
                [source_line_y, source_line_y],
                color=color,
                linewidth=2.10,
                zorder=5,
            )
            ax_bottom.text(
                position,
                source_line_y * 1.32,
                "none",
                ha="center",
                va="bottom",
                fontsize=9.2,
                fontweight="bold",
                color="0.32",
            )
            continue

        payload_value = float(payload_by_method[method])
        ratio = payload_value / dmet_payload

        ax_bottom.vlines(
            position,
            line_start,
            payload_value,
            color=color,
            linewidth=5.0,
            alpha=0.72,
            zorder=3,
        )
        ax_bottom.scatter(
            position,
            payload_value,
            s=58,
            marker="o",
            facecolors=color,
            edgecolors="black",
            linewidths=0.72,
            zorder=6,
        )

        if payload_value > 1500:
            text_y = payload_value / 1.45
            vertical_alignment = "top"
        else:
            text_y = payload_value * 1.60
            vertical_alignment = "bottom"

        if method == "dmet_direct":
            label_text = f"{payload_value:.2f} MiB\n1×"
        elif ratio < 20:
            label_text = f"{payload_value:.2f} MiB\n{ratio:.1f}×"
        elif ratio < 1000:
            label_text = f"{payload_value:.0f} MiB\n{ratio:.0f}×"
        else:
            label_text = f"{payload_value:.0f} MiB\n{ratio:.0f}×"

        ax_bottom.text(
            position,
            text_y,
            label_text,
            ha="center",
            va=vertical_alignment,
            fontsize=9.0,
            color="0.20",
            fontweight="bold",
            zorder=10,
            bbox={
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.78,
                "pad": 0.6,
            },
        )

    ax_bottom.set_ylabel(
        "Payload\n(MiB, log)",
        fontsize=11.0,
        fontweight="bold",
        labelpad=8,
    )

    preferred_ticks = [0.5, 5, 50, 500, 2500]
    visible_ticks = [
        tick
        for tick in preferred_ticks
        if lower_comm_limit <= tick <= upper_comm_limit
    ]
    if visible_ticks:
        ax_bottom.set_yticks(visible_ticks)
        ax_bottom.set_yticklabels(
            [f"{tick:g}" for tick in visible_ticks]
        )

    ax_bottom.grid(
        True,
        axis="y",
        which="major",
        alpha=0.23,
        linewidth=0.65,
    )
    ax_bottom.grid(
        True,
        axis="y",
        which="minor",
        alpha=0.10,
        linewidth=0.30,
    )
    ax_bottom.set_xticks(np.arange(len(figure_order)))
    ax_bottom.set_xticklabels(
        [method_labels[method] for method in figure_order],
        fontsize=10.8,
        fontweight="bold",
    )
    ax_bottom.tick_params(axis="x", length=0, pad=6)

    for spine in ax_bottom.spines.values():
        spine.set_linewidth(1.10)
        spine.set_color("black")
    for tick in ax_bottom.get_yticklabels():
        tick.set_fontweight("bold")

    fig.subplots_adjust(
        left=0.085,
        right=0.985,
        top=0.885,
        bottom=0.145,
    )

    for extension in ("pdf", "png", "svg"):
        output_path = output_stem.with_suffix(f".{extension}")
        fig.savefig(
            output_path,
            dpi=600,
            bbox_inches="tight",
        )
        print("saved:", output_path)

    plt.show()
    plt.close(fig)

    caption_text = (
        "Seed-level robustness and communication footprint. "
        "Panel A shows remote Macro F1 (%) across random seeds for each "
        "method. Seed-specific markers denote individual seed-level "
        "results, diamond markers denote the across-seed mean, and "
        "min--max bars denote the observed seed range. Panel B reports "
        "the total logical protocol communication payload on a logarithmic "
        "scale, aligned to the same methods; ratios are shown relative to "
        "DMET-SSTKG. The DMET-SSTKG payload follows the logical online "
        "query--response definition of 528 bytes per uncertainty-selected "
        "motif, with the one-time hub object excluded. Shading highlights "
        "the source-only reference and DMET-SSTKG."
    )

    caption_path = output_stem.parent / (
        output_stem.name + "_caption.txt"
    )
    caption_path.write_text(caption_text, encoding="utf-8")
    print("saved:", caption_path)

    return summary_df


def save_comparative_outputs(
    benchmark: BenchmarkOutputs,
) -> dict[str, Path]:
    output_dir = Path(benchmark.config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    summary = build_comparative_summary(benchmark)

    seed_results = benchmark.seed_metrics.copy()
    seed_results["macro_f1_percent"] = (
        100.0 * seed_results["macro_f1"]
    )
    seed_results["accuracy_percent"] = (
        100.0 * seed_results["accuracy"]
    )

    communication = benchmark.communication.copy()
    communication["total_protocol_mib"] = (
        communication["total_protocol_bytes"] / (1024.0**2)
    )

    figure_stem = (
        output_dir
        / "Comparative_SeedRobustness_PayloadFootprint"
    )

    paths = {
        "summary_csv": (
            output_dir / "Comparative_Remote_Performance.csv"
        ),
        "summary_tex": (
            output_dir / "Comparative_Remote_Performance.tex"
        ),
        "seed_csv": (
            output_dir / "Comparative_SeedLevel_Results.csv"
        ),
        "communication_csv": (
            output_dir / "Comparative_Communication_Payload.csv"
        ),
        "figure_summary_csv": (
            output_dir
            / "Comparative_SeedRobustness_PayloadFootprint_summary.csv"
        ),
        "figure_pdf": figure_stem.with_suffix(".pdf"),
        "figure_png": figure_stem.with_suffix(".png"),
        "figure_svg": figure_stem.with_suffix(".svg"),
        "figure_caption": (
            output_dir
            / "Comparative_SeedRobustness_PayloadFootprint_caption.txt"
        ),
        "manifest_json": (
            output_dir / "Comparative_RunManifest.json"
        ),
    }

    summary.to_csv(paths["summary_csv"], index=False)
    write_comparative_latex(summary, paths["summary_tex"])
    seed_results.to_csv(paths["seed_csv"], index=False)
    communication.to_csv(paths["communication_csv"], index=False)

    figure_summary = make_seed_payload_figure(
        seed_results,
        communication,
        figure_stem,
    )

    # The figure routine writes this file; save again explicitly to
    # preserve the output contract even if the routine is refactored.
    figure_summary.to_csv(
        paths["figure_summary_csv"],
        index=False,
    )

    manifest = {
        "status": "COMPLETE",
        "inputs": {
            "source_binary_cache": str(
                benchmark.config.source_binary_cache
            ),
            "remote_binary_cache": str(
                benchmark.config.remote_binary_cache
            ),
            "source_split_npz": str(
                benchmark.config.source_split_npz
            ),
            "base_motif_checkpoint": str(
                benchmark.config.base_motif_checkpoint
            ),
            "dmet_hub": str(
                benchmark.config.dmet_hub
            ),
        },
        "protocol": {
            "seeds": list(benchmark.config.seeds),
            "selection_seed": (
                benchmark.config.selection_seed
            ),
            "num_clients": benchmark.config.num_clients,
            "communication_rounds": (
                benchmark.config.rounds
            ),
            "hyperparameter_mode": (
                benchmark.config.hyperparameter_mode
            ),
            "remote_label_use": (
                "evaluation only after public predictions"
            ),
            "dmet_payload": (
                "512-byte float32 query + "
                "16-byte float16 response"
            ),
        },
        "locked_candidates": benchmark.locked_candidates,
        "outputs": {
            key: str(path)
            for key, path in paths.items()
        },
    }

    paths["manifest_json"].write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    display(
        summary[
            [
                "display_name",
                "macro_f1_percent",
                "accuracy_percent",
                "macro_f1_gain_vs_source_pp",
                "communication_payload_mib",
            ]
        ]
    )

    return paths


comparative_artifacts = save_comparative_outputs(benchmark)

print("\nGenerated artifacts:")
for name, path in comparative_artifacts.items():
    print(f"- {name}: {path}")

## References

[1] McMahan, B., Moore, E., Ramage, D., Hampson, S., & y Arcas, B. A. (2017). Communication-efficient learning of deep networks from decentralized data. *Proceedings of the 20th International Conference on Artificial Intelligence and Statistics*, 1273–1282.

[2] Karimireddy, S. P., Kale, S., Mohri, M., Reddi, S. J., Stich, S. U., & Suresh, A. T. (2020). SCAFFOLD: Stochastic controlled averaging for federated learning. *Proceedings of the 37th International Conference on Machine Learning*, 5132–5143.

[3] Tan, Y., Long, G., Liu, L., Zhou, T., Lu, Q., Jiang, J., & Zhang, C. (2022). FedProto: Federated prototype learning across heterogeneous clients. *Proceedings of the AAAI Conference on Artificial Intelligence, 36*(8), 8432–8440.

[4] Zhang, J., Liu, Y., Hua, Y., & Cao, J. (2024). FedTGP: Trainable global prototypes with adaptive-margin-enhanced contrastive learning for data and model heterogeneity in federated learning. *Proceedings of the AAAI Conference on Artificial Intelligence, 38*(15), 16768–16776.

[5] Lian, X., Zhang, C., Zhang, H., Hsieh, C.-J., Zhang, W., & Liu, J. (2017). Can decentralized algorithms outperform centralized algorithms? A case study for decentralized parallel stochastic gradient descent. *Advances in Neural Information Processing Systems, 30*.